# Project Goals

* Explore what happened to stocks of various risk factor exposures and industry exposures around the 2008 Global Financial Crisis (GFC)

About the GFC:

* The 2008 Global Financial Crisis (GFC) emerged from the collapse of the U.S. housing market and the subsequent unraveling of mortgage-backed securities (MBS) and collateralized debt obligations (CDOs). As home prices declined and foreclosures increased, these complex financial instruments, which were widely held by banks and financial institutions, began losing value rapidly.

# Barra Risk Model

The Barra risk model is a multi-factor model used to predict risk and returns for portfolios of securities like stocks and bonds. It helps in:

* Understanding Risk Exposures: By decomposing risk into specific factors, it allows investors to understand the sources of their portfolio's risk and return. These risk exposures may come from macroeconomic factors or industry sector memberships etc.

* Portfolio Optimization: By analyzing risk and return, investors can optimize portfolios by adjusting exposures to various risk factors. This would entail increasing exposure to high-growth factors while reducing exposures to sources of risk.

* Performance Attribution: The Barra model is also useful to study how specific risk factors affect portfolio returns. Through decomposing portfolio performance into factor-level contributions, investors can get a better understanding of drivers of past performance and potential areas for improvement.

## Applications
The Barra risk model is employed by:
* Portfolio Managers to measure and manage the risk of their portfolios.
* Quantitative Analysts to develop investment strategies.
* Risk Managers to control the risk of financial institutions.


## Data Handbook
* Reference: the Barra US Equity Model (USE3) handbook (industry and sector definitions, factor descriptions).

# Barra Data from Dec 2007
* Before the Global Financial Crisis of 2008

The Barra USE3 files (`USE3L0712.RSK` for December 2007, `USE3L0812.RSK` for December 2008) are licensed and are not included in this repository; outputs that showed company-level rows have been cleared. See the README for the file layout.

In [ ]:
# Data download removed: the Barra USE3 file is licensed and not redistributed (see README).

In [ ]:
! ls

In [ ]:
! head /content/USE3L0712.RSK

# Barra Data from Dec 2008
* After the Global Financial Crisis of 2008

In [ ]:
# Data download removed: the Barra USE3 file is licensed and not redistributed (see README).

In [ ]:
! head USE3L0812.RSK

## Extract

* Comma seperated values file
* Ignore first row = date
* Second row = header row


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
barra_df = pd.read_csv('USE3L0712.RSK', skiprows=1)

In [ ]:
barra_df

In [ ]:
# Data exploration
barra_df.head()

In [ ]:
barra_df.tail()

In [ ]:
barra_df.info()

In [ ]:
barra_df.columns

In [ ]:
barra_df.describe()

In [ ]:
barra_df[['HBTA',]].boxplot()

In [ ]:
barra_df.HBTA.min()

In [ ]:

barra_df[['BETA']].boxplot()

In [ ]:
# barra_df.loc[row,col] # scalar
# barra_df.loc[:,col] # Series of all rows
# barra_df.loc[row,:] # Series of all cols
# barra_df.loc[:,'SIZE'] # ERROR

barra_df.columns # Has extra spaces
barra_df.columns # Has '%' to be removed


## Transform (Data Cleaning)

Issues:
1. Column names have extra spaces and some have '%'
2. Some text columns have values with extra spaces
3. Some values are -999 when they should NaN (not a number)

In [ ]:
'SIZE    '

In [ ]:
'SIZE    '.strip() # Solves 1 for one string


In [ ]:
'WGT1%'.replace('%','_PCT')

In [ ]:
barra_df.columns = barra_df.columns.str.strip() # Solves 1
barra_df.columns = barra_df.columns.str.replace('%','_PCT') # Solves 1
barra_df.columns

In [ ]:
# barra_df.set_index('TICKER').loc['AAPL'] # Key Error!
barra_df.set_index('TICKER').index

In [ ]:
# Select only columns that contain strings
barra_df.select_dtypes('object')

In [ ]:
for col in barra_df.select_dtypes(include='object'): # Solves 2
  print(col)
  barra_df[col] = barra_df[col].str.strip()



In [ ]:
barra_df.set_index('TICKER').loc['AAPL']

In [ ]:
barra_df = barra_df.replace(-999, np.nan) # Solves Issue 3

## Load
* Now we can load into SQL


# Understanding SQL, SQLAlchemy, and Data Types

## What is SQL?
SQL (Structured Query Language) is a standardized programming language used for managing and manipulating relational databases. A relational database organizes data into tables with rows and columns, similar to spreadsheets. However, unlike spreadsheets, relational databases can:
- Handle much larger amounts of data efficiently
- Enforce data integrity and consistency
- Allow multiple users to access and modify data simultaneously
- Enable complex queries across multiple tables
- Provide better security and access control

## What is SQLAlchemy?
SQLAlchemy is a Python library that serves as an Object Relational Mapper (ORM). It acts as a bridge between Python code and databases, allowing you to:
- Write Python code instead of SQL queries
- Work with databases using Python objects
- Support multiple database engines (SQLite, PostgreSQL, MySQL, etc.)
- Handle database connections and transactions
- Provide data type mapping between Python and SQL

## Why Define Data Types?
In our code, we defined specific data types for each column (like `String(50)`, `Float`, `Integer`). This is important for several reasons:

1. **Data Integrity**:
   - Prevents storing wrong data types (e.g., text in a number column)
   - Ensures consistent data format across all records
   - Helps catch errors early in the data entry process

2. **Storage Efficiency**:
   - Different data types use different amounts of storage space
   - Proper type selection optimizes database size
   - Example: Storing numbers as integers instead of strings saves space

3. **Query Performance**:
   - Correct data types enable proper indexing
   - Improves search and sort operations
   - Allows for proper numerical comparisons

4. **Memory Management**:
   ```python
   'BARRID': String(50)  # Limits string length to 50 characters
   'PRICE': Float        # Stores decimal numbers efficiently
   'SAP500': Integer     # Only stores whole numbers
   ```

## Common SQLAlchemy Data Types:
```python
from sqlalchemy.types import Integer, Float, String, Date, Boolean

# Common mappings
dtype_dict = {
    'id': Integer,           # Whole numbers
    'price': Float,          # Decimal numbers
    'name': String(100),     # Text with max length
    'date': Date,           # Date values
    'active': Boolean       # True/False values
}
```

## Best Practices:
1. **Choose Appropriate Lengths**:
   - Don't make strings unnecessarily long
   - Consider future data requirements
   - Balance between flexibility and efficiency

2. **Number Types**:
   - Use `Integer` for whole numbers
   - Use `Float` for decimal numbers
   - Consider precision requirements

3. **Special Types**:
   - Use `Date` for dates instead of strings
   - Use `Boolean` for true/false values
   - Use `Enum` for fixed sets of values

4. **Documentation**:
   - Comment why specific types were chosen
   - Document any constraints or assumptions
   - Keep track of any type changes



In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.types import Date, Float, String, Integer, Boolean

barra_df.info()

In [ ]:
# Create SQLite database engine
engine = create_engine('sqlite:///barra_database.db')

# Define data types for SQL columns
dtype_dict = {
    # String columns
    'BARRID': String(50),
    'TICKER': String(10),
    'CUSIP': String(10),
    'NAME': String(100),
    'INDNAME1': String(100),
    'INDNAME2': String(100),
    'INDNAME3': String(100),
    'INDNAME4': String(100),
    'INDNAME5': String(100),

    # Float columns (all numeric factors)
    'HBTA': Float,
    'BETA': Float,
    'SRISK%': Float,
    'TRISK%': Float,
    'VOLTILTY': Float,
    'MOMENTUM': Float,
    'SIZE': Float,
    'SIZENONL': Float,
    'TRADEACT': Float,
    'GROWTH': Float,
    'EARNYLD': Float,
    'VALUE': Float,
    'EARNVAR': Float,
    'LEVERAGE': Float,
    'CURRSEN': Float,
    'YIELD': Float,
    'NONESTU': Float,
    'PRICE': Float,
    'CAPITALIZATION': Float,
    'YLD%': Float,

    # Integer columns
    'IND1': Integer,
    'WGT1%': Integer,
    'IND2': Integer,
    'WGT2%': Integer,
    'IND3': Integer,
    'WGT3%': Integer,
    'IND4': Integer,
    'WGT4%': Integer,
    'IND5': Integer,
    'WGT5%': Integer,
    'SAP500': Integer,
    'SAPVAL': Integer,
    'SAPGRO': Integer,
    'MIDCAP': Integer,
    'MIDVAL': Integer,
    'MIDGRO': Integer,
    'SC600': Integer,
    'SCVAL': Integer,
    'SCGRO': Integer,
    'E3ESTU': Integer
}

# Load DataFrame to SQL
barra_df.to_sql('barra_factors',
                engine,
                if_exists='replace',
                index=False,
                dtype=dtype_dict)


In [ ]:
# Question assumptions
barra_df.TICKER.str.len().describe() # Actually only up to 6 characters. But will 2008 need more?

In [ ]:
barra_df.NAME.str.len().describe() # Up to 32 characters.

In [ ]:
# Hint - file is fixed width, so you can grab the original widths

pd.read_csv('USE3L0712.RSK', skiprows=1).select_dtypes('object').map(len).describe()

In [ ]:
# To do - these should be Boolean and not Integer!
barra_df.SAP500.describe()

# SQL Queries for Barra Factor Analysis

## 1. Basic Count Query
```sql
SELECT COUNT(*) as total_records
FROM barra_factors;
```
This query gives us the total number of records in our database. It's useful for verifying that all data was loaded correctly.



In [ ]:
# 1. Basic count of records
count_query = """
SELECT COUNT(*) as total_records
FROM barra_factors;
"""
print("Total Records:")
print(pd.read_sql(count_query, engine)) # This is from the SQL database, not DataFrame


In [ ]:
len(barra_df) # matches DataFrame


## 2. Summary Statistics Query
```sql
SELECT
    COUNT(*) as count,
    AVG(BETA) as avg_beta,
    AVG(SIZE) as avg_size,
    AVG(MOMENTUM) as avg_momentum,
    AVG(VALUE) as avg_value
FROM barra_factors;
```
This query calculates average values for key Barra factors. Common aggregate functions include:
- `AVG()`: Average value
- `STDDEV()`: Standard deviation
- `MIN()`: Minimum value
- `MAX()`: Maximum value


In [ ]:
# 2. Summary statistics for key factors
stats_query = """
SELECT
    COUNT(*) as count,
    AVG(BETA) as avg_beta,
    AVG(SIZE) as avg_size,
    AVG(MOMENTUM) as avg_momentum,
    AVG(VALUE) as avg_value
FROM barra_factors;
"""

print("\nSummary Statistics:")
print(pd.read_sql(stats_query, engine))


# GROUP BY and OVER

```sql
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap
FROM barra_factors
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
```
This query demonstrates:
- `GROUP BY`: Aggregating data by category
- `ORDER BY`: Sorting results
- `LIMIT`: Restricting number of results


GROUP BY
* **Collapses** rows into summary statistics
* Returns one row per group
* Loses individual row details


In [ ]:
# 3. Distribution of industries (top 5)
industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap
FROM barra_factors
GROUP BY INDNAME1;
"""
print("\nIndustry Distribution:")
print(pd.read_sql(industry_query, engine))


In [ ]:
# 3. Distribution of industries (top 5)
industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap
FROM barra_factors
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""
print("\nIndustry Distribution:")
print(pd.read_sql(industry_query, engine))


In [ ]:
# 3. Distribution of industries (top 5)
industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap
FROM barra_factors
GROUP BY INDNAME1
ORDER BY avg_cap DESC
LIMIT 5;
"""
print("\nIndustry Distribution:")
print(pd.read_sql(industry_query, engine))


## Window Functions

* **Maintains** all rows
* Adds calculations to each row
* Can see individual values alongside aggregates

In [ ]:
print("\nWHERE LIQUID STOCKS:")
print(pd.read_sql('''
SELECT
    NAME
FROM barra_factors
WHERE E3ESTU = 1 AND SIZE>1 --- Liquid US Stocks within ESTUNI and size greater than 1 (this is a comment)
ORDER BY SIZE DESC;
''', engine))


In [ ]:
print("\nIndustry Distribution:")
print(pd.read_sql('''
SELECT
    NAME,
    INDNAME1,
    LEVERAGE,
    AVG(LEVERAGE) OVER (PARTITION BY INDNAME1) as ind_avg_lev,
    LEVERAGE - AVG(LEVERAGE) OVER (PARTITION BY INDNAME1) as diff_from_avg
FROM barra_factors
WHERE E3ESTU = 1 AND SIZE>1 --- Liquid US Stocks within ESTUNI and size greater than 1
ORDER BY SIZE DESC;
''', engine))


In [ ]:
print("\nIndustry Distribution:")
print(pd.read_sql('''
SELECT
    NAME,
    LEVERAGE,
    AVG(LEVERAGE) OVER (PARTITION BY INDNAME1) as ind_avg_lev,
    LEVERAGE - AVG(LEVERAGE) OVER (PARTITION BY INDNAME1) as diff_from_avg
FROM barra_factors
WHERE E3ESTU = 1 AND SIZE>-1 AND INDNAME1 = 'BANKS' --- Liquid US BANKS within ESTUNI and size greater than -1
ORDER BY SIZE DESC;
''', engine))


 NATIONAL CITY CORP (Much higher leverage than ind avg)
 * No, National City Corp. is no longer an independent bank. It was acquired by PNC Financial Services Group on December 31, 2008, following significant losses during the subprime mortgage crisis. National City branches were rebranded and integrated into PNC, and the corporation ceased to exist as a separate entity.

# DISTINCT
To get unique industry names (INDNAME1), you can use the DISTINCT keyword in SQL. Here's how:


In [ ]:
# Query to get unique industry names
unique_industry_query = """
SELECT INDNAME1 as 'Industry Name'
FROM barra_factors
ORDER BY INDNAME1;
"""

# Execute and display results
unique_industries = pd.read_sql(unique_industry_query, engine)
print("\nUnique Industries in Dataset:")
print(unique_industries)

# BANKS
# SECASSET

In [ ]:
# Query to get unique industry names
unique_industry_query = """
SELECT DISTINCT INDNAME1 as 'Industry Name'
FROM barra_factors
ORDER BY INDNAME1;
"""

# Execute and display results
unique_industries = pd.read_sql(unique_industry_query, engine)
print("\nUnique Industries in Dataset:")
print(unique_industries)

# BANKS
# SECASSET

In [ ]:
# Query to show how many unique industries exist
count_industries_query = """
SELECT COUNT(DISTINCT INDNAME1) as 'Number of Unique Industries'
FROM barra_factors;
"""

# Execute and display count
industry_count = pd.read_sql(count_industries_query, engine)
print("\nTotal Number of Unique Industries:")
print(industry_count)

# Understanding SQL Filtering with WHERE and AND

Let's explore how to filter data in SQL using a practical example with our Barra dataset. We'll look at how to find specific industries and add additional conditions.

## Basic Industry Query
First, let's look at our base query:
```python
# Basic industry distribution
basic_industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap
FROM barra_factors
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""

pd.read_sql(basic_industry_query, engine)
```

## Adding WHERE Clause
Now let's add filtering to focus on large-cap stocks:
```python
# Adding WHERE to filter results
filtered_industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap
FROM barra_factors
WHERE CAPITALIZATION > 1000000000  -- Filter for large cap stocks > $1bn
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""

pd.read_sql(filtered_industry_query, engine)
```

## Using Multiple Conditions with AND
Let's add more conditions to find large-cap stocks with positive momentum:
```python
# Using WHERE with multiple conditions
complex_industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap,
    AVG(MOMENTUM) as avg_momentum
FROM barra_factors
WHERE CAPITALIZATION > 1000000000     -- First condition
    AND MOMENTUM > 0               -- Second condition
    AND NONESTU = 0               -- Third condition
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""

pd.read_sql(complex_industry_query, engine)
```

## Understanding SQL Filtering
1. **WHERE Clause**:
   - Acts like a filter on your data
   - Goes after the FROM statement
   - Only rows that meet the condition are included
   - Example: `WHERE CAPITALIZATION > 1000000` only shows large-cap stocks

2. **AND Operator**:
   - Combines multiple conditions
   - ALL conditions must be true for the row to be included
   - Think of it like a series of filters
   - Example: `WHERE CAPITALIZATION > 1000000 AND MOMENTUM > 0` shows only large-cap stocks with positive momentum

3. **Order of Operations**:
   ```sql
   SELECT columns           -- 3. Select specific columns
   FROM table              -- 1. Start with table
   WHERE conditions        -- 2. Filter rows
   GROUP BY column         -- 4. Group results
   ORDER BY column         -- 5. Sort results
   LIMIT number;           -- 6. Limit number of rows
   ```


In [ ]:
# Using WHERE with multiple conditions
complex_industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap,
    AVG(MOMENTUM) as avg_momentum
FROM barra_factors
WHERE CAPITALIZATION > 1000000000     -- First condition
    AND MOMENTUM > 0               -- Second condition
    AND NONESTU = 0               -- Third condition
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""

pd.read_sql(complex_industry_query, engine)

# Understanding the IN Operator in SQL

## What is the IN Operator?
The IN operator is a powerful SQL feature that allows you to check if a value matches any value in a list of values. It's similar to saying "where x equals this OR that OR that..."

## Basic Syntax
```sql
WHERE column_name IN (value1, value2, value3)
```

## Comparison with OR
Consider these two equivalent queries:

Using OR:
```sql
WHERE INDNAME1 = 'BANKS'
   OR INDNAME1 = 'THRIFTS'
   OR INDNAME1 = 'SECASSET'
```

Using IN:
```sql
WHERE INDNAME1 IN ('BANKS', 'THRIFTS', 'SECASSET')
```

The IN operator is:
- More concise
- Easier to read
- Less prone to errors
- Often more efficient

## Real Example with Barra Data
```python
# Finding all financial companies
financial_query = """
SELECT
    INDNAME1 as Industry,
    COUNT(*) as Count
FROM barra_factors
WHERE INDNAME1 IN ('BANKS', 'THRIFTS', 'SECASSET', 'FINSVCS')
GROUP BY INDNAME1;
"""
```

## Advanced Features
1. **Can use with subqueries:**
```sql
WHERE MOMENTUM IN (SELECT TOP 10 MOMENTUM FROM barra_factors)
```

2. **Can use NOT IN:**
```sql
WHERE INDNAME1 NOT IN ('BANKS', 'THRIFTS')
```

3. **Works with multiple types:**
```sql
WHERE value IN (100, 200, 300)  -- numbers
WHERE status IN ('active', 'pending')  -- strings
```


Hint:
Page 84 of PDF:

Financial Sector consists of
- Financial Life & Health Insurance
- Property & Casualty Insurance
- Banks
- Thrifts
- Securities & Asset Management
- Financial Services

In [ ]:
# Finding all financial companies
financial_query = """
SELECT
    INDNAME1 as Industry,
    COUNT(*) as Count
FROM barra_factors
WHERE INDNAME1 IN ('BANKS', 'THRIFTS', 'SECASSET', 'FINSVCS') AND NONESTU = 0
GROUP BY INDNAME1;
"""


# Execute and show results
pd.read_sql(financial_query, engine)


# Understanding UNION in SQL

## What is UNION?
UNION is a SQL operator that combines the results of two or more SELECT statements into a single result set. It's particularly useful when you want to:
- Compare metrics across different groups
- Combine similar data from different categories
- Create summary tables with labeled sections

## Basic Syntax
```sql
SELECT column1, column2 FROM table1
UNION
SELECT column1, column2 FROM table2
```

## Key Rules for UNION
1. Each SELECT statement must have the same number of columns
2. Corresponding columns must have similar data types
3. The column names from the first SELECT statement are used as the column names for the result set

## Practical Example: Comparing Utilities
```sql
SELECT
    'Gas' as Sector,            -- Creates label 'Gas'
    AVG(CAPITALIZATION)/1e9 as Avg_MarketCap_Billions
FROM barra_factors
WHERE INDNAME1 IN ('GASUTIL')
AND NONESTU = 0
UNION
SELECT
    'Electric' as Sector,       -- Creates label 'Electric'
    AVG(CAPITALIZATION)/1e9 as Avg_MarketCap_Billions
FROM barra_factors
WHERE INDNAME1 IN ('ELECUTIL')
AND NONESTU = 0;
```

In this example:
- First query calculates average market cap for gas utilities
- Second query calculates average market cap for electric utilities
- UNION combines them into one table with labeled sectors
- Both queries must return same number of columns (2 in this case)
- Both queries must have compatible types (text and number)


In [ ]:
# Example template for sector comparison
comparison_query = """
SELECT
    'Gas' as Sector,
    AVG(CAPITALIZATION)/1e9 as Avg_MarketCap_Billions
FROM barra_factors
WHERE INDNAME1 IN ('GASUTIL')
AND NONESTU = 0;
"""

pd.read_sql(comparison_query, engine)

In [ ]:
# Example template for sector comparison
comparison_query = """
SELECT
    'Electric' as Sector,
    AVG(CAPITALIZATION)/1e9 as Avg_MarketCap_Billions
FROM barra_factors
WHERE INDNAME1 IN ('ELECUTIL')
AND NONESTU = 0;
"""

pd.read_sql(comparison_query, engine)

In [ ]:
# Example template for sector comparison
comparison_query = """
SELECT
    'Gas' as Sector,
    AVG(CAPITALIZATION)/1e9 as Avg_MarketCap_Billions
FROM barra_factors
WHERE INDNAME1 IN ('GASUTIL')
AND NONESTU = 0
UNION
SELECT
    'Electric' as Sector,
    AVG(CAPITALIZATION)/1e9 as Avg_MarketCap_Billions
FROM barra_factors
WHERE INDNAME1 IN ('ELECUTIL')
AND NONESTU = 0;
"""

pd.read_sql(comparison_query, engine)

# Student Research Questions: Analyzing Sectors During the GFC

## Important Note
For all queries, remember to include `AND NONESTU = 0 AND SIZE>0` to restrict analysis to the larger companies within the estimation universe (i.e. no foreign companies). This ensures we're working with actively traded, liquid stocks that Barra uses in their factor model.

## Core Research Questions

1. **Risk Characteristics**
   ```sql
   -- Always include: AND NONESTU = 0
   SELECT
       AVG(BETA) as avg_beta,
       AVG(VOLTILTY) as avg_volatility
   FROM barra_factors
   WHERE INDNAME1 IN ('BANKS', 'THRIFTS', 'SECASSET', 'FINSVCS')
   AND NONESTU = 0;   -- Critical for estimation universe
   ```
   * How do betas compare between financial and technology sectors?
   * Which sector has higher volatility?
   * Is systematic risk (SRISK%) significantly different?

2. **Size and Trading**
   * Compare market capitalization between two sectors (remember NONESTU = 0)
   * Is trading activity (TRADEACT) different?
   * Do certain sectors show more momentum?

## Advanced Research Questions

3. **Leverage Analysis**
   ### Question 3.1: High Leverage Financial Institutions
   * Find top 10 banks by leverage ratio
   * Compare their market caps and volatility
   * Analyze relationship between leverage and other risk metrics

   ### Question 3.2: Industry Leverage Comparison
   * Rank industries by average leverage
   * Compare financial subsectors
   * Consider size effects in leverage analysis

4. **Index Membership**
   ### Question 4.1: Bank Index Distribution
   * Analyze distribution across SAP500, SAPVAL, SAPGRO
   * Calculate percentage of banks in each index
   * Compare with other sectors
   * Given that banks crashed in 2008, which index is most impacted?

   ### Question 4.2: Value vs Growth Classification
   * Study SAPVAL vs SAPGRO membership patterns
   * Examine MIDVAL vs MIDGRO distribution
   * Analyze SCVAL vs SCGRO composition
   * E.g. Do companies in SAPVAL have logically a higher SIZE and VALUE and lower GROWTH values compared to companies in SCVAL?

5. **Multi-Industry Analysis**
   ### Question 5.1: Industry Overlap
   * Identify firms with non-zero WGT2%
   * Compare diversified vs focused firms
   * Analyze impact on risk metrics

   ### Question 5.2: Yield Analysis
   * Find highest-yield financial institutions
   * Study yield-risk relationships
   * Examine yield stability across size groups

## Tips for Analysis
1. Always start queries with checking how many stocks are in each sector (N) in the estimation universe
2. Consider size differences when comparing sectors
3. Look at both averages and distributions of factors
4. Consider combining multiple factors in your analysis
5. Think about appropriate presentation formats (averages, rankings, etc.)
6. Remember to consider market cap effects in all analyses


In [ ]:
sectors = {
    'FINANCIAL': ['BANKS', 'THRIFTS', 'SECASSET', 'FINSVCS', 'PRPTYINS', 'LIFEINS'],
    'TECH_COMM': ['SEMICOND', 'CMPTRSW', 'CMPTRHW', 'INTERNET', 'INFOSVCS',
                  'ELECEQP', 'TELEPHON', 'WIRELESS', 'MEDIA', 'PUBLISH'],
    'REALESTATE_UTIL': ['EQTYREIT', 'ELECUTIL', 'GASUTIL'],
    'HEALTHCARE': ['MEDPRODS', 'BIOTECH', 'DRUGS', 'MEDPROVR'],
    'CONS_DISCR': ['SPLTYRET', 'RESTRNTS', 'CLOTHING', 'APPAREL', 'DEPTSTOR',
                   'HOTELS', 'ENTRTAIN', 'LEISURE', 'MOTORVEH', 'CONSDUR'],
    'CONS_STAPLES': ['GROCERY', 'FOODBEV', 'HOMEPROD', 'ALCOHOL', 'TOBACCO'],
    'INDUSTRIAL': ['CONSTRUC', 'INDPART', 'INDSVCS', 'HEAVYELC', 'HEAVYMCH',
                  'DEFAERO', 'ENVSVCS'],
    'BASIC_MAT': ['CHEMICAL', 'FOREST', 'GOLD'],
    'ENERGY': ['ENGYRES', 'OILSVCS', 'OILREF', 'MINING'],
    'TRANSPORT': ['AIRLINES', 'RAILROAD', 'TRUCKFRT']
}


In [ ]:
# Pause class
# Students to do this in-class exercise -- as many SQL queries that they believe are informative.
# and extend for their project submissions.


In [ ]:
# Question compare stocks where INDNAME1 = 'BANKS' or INDNAME2 = 'BANKS'

comparison_query = """
SELECT
    'PrimaryBank' as Sector,
    AVG(CAPITALIZATION)/1e9 as Avg_MarketCap_Billions,
    AVG(LEVERAGE) as Leverage
FROM barra_factors
WHERE INDNAME1 IN ('BANKS')
AND NONESTU = 0
UNION
SELECT
    'SecondaryBank' as Sector,
    AVG(CAPITALIZATION)/1e9 as Avg_MarketCap_Billions,
    AVG(LEVERAGE) as Leverage
FROM barra_factors
WHERE INDNAME2 IN ('BANKS') OR INDNAME3 IN ('BANKS') OR INDNAME4 IN ('BANKS') OR INDNAME5 IN ('BANKS')
AND NONESTU = 0;
"""

pd.read_sql(comparison_query, engine)

In [ ]:
# Question - Do financials have a higher BETA/VOLATILTY on average than technology?
pd.read_sql(
'''
SELECT
    'Financials' as Sector,
    AVG(BETA) as avg_beta,
    AVG(VOLTILTY) as avg_volatility
FROM barra_factors
WHERE INDNAME1 IN ('BANKS', 'THRIFTS', 'SECASSET', 'FINSVCS', 'PRPTYINS', 'LIFEINS')
AND NONESTU = 0 AND SIZE>0
UNION
SELECT
    'Tech' as Sector,
    AVG(BETA) as avg_beta,
    AVG(VOLTILTY) as avg_volatility
FROM barra_factors
WHERE INDNAME1 IN ('SEMICOND', 'CMPTRSW', 'CMPTRHW', 'INTERNET', 'INFOSVCS')
AND NONESTU = 0 AND SIZE>0;   -- Critical for estimation universe
''',engine)


In [ ]:
# Question - Do financials have a higher LEVERAGE on average than technology?
pd.read_sql(
'''
SELECT
    'Financials' as Sector,
    AVG(LEVERAGE) as avg_lev
FROM barra_factors
WHERE INDNAME1 IN ('BANKS', 'THRIFTS', 'SECASSET', 'FINSVCS', 'PRPTYINS', 'LIFEINS')
AND NONESTU = 0 AND SIZE>0
UNION
SELECT
    'Tech' as Sector,
    AVG(LEVERAGE) as avg_lev
FROM barra_factors
WHERE INDNAME1 IN ('SEMICOND', 'CMPTRSW', 'CMPTRHW', 'INTERNET', 'INFOSVCS')
AND NONESTU = 0 AND SIZE>0;   -- Critical for estimation universe
''',engine)


# CTEs, Views, and Subqueries: Three Ways to Organize SQL Logic

## Overview
SQL offers three powerful techniques for organizing complex queries and reusing calculations: CTEs (Common Table Expressions), Views, and Subqueries. Each serves different purposes and has distinct advantages.

## 1. Common Table Expressions (CTEs) - WITH Clause

### What is a CTE?
A CTE creates a temporary named result set that exists only for the duration of a single query. Think of it as a variable that holds query results temporarily.

### Syntax
```sql
WITH temp_name AS (
    -- Your query here
    SELECT ...
)
SELECT * FROM temp_name;
```

### When to Use CTEs
- **Complex multi-step calculations** within a single query
- **Need to reference the same result multiple times** in one query
- **Want readable, self-documenting code** with named intermediate steps
- **Recursive operations** (like hierarchical data traversal)

### Example
```sql
WITH momentum_stats AS (
    SELECT AVG(MOMENTUM) as mean, STDDEV(MOMENTUM) as stddev
    FROM barra_factors WHERE NONESTU = 0
)
SELECT NAME, MOMENTUM,
       (MOMENTUM - mean) / stddev as z_score
FROM barra_factors, momentum_stats
WHERE MOMENTUM > mean + stddev;
```

## 2. Views - Persistent Named Queries

### What is a View?
A View is a stored query definition that acts like a virtual table. Unlike CTEs, views persist in the database and can be used across multiple queries and sessions.

### Syntax
```sql
CREATE VIEW view_name AS
SELECT ...;

-- Use it like a table
SELECT * FROM view_name;
```

### When to Use Views
- **Frequently used complex queries** that multiple users/queries need
- **Simplifying database access** for non-technical users
- **Security** - restrict access to specific columns/rows
- **Consistent business logic** across applications

### Example
```sql
CREATE VIEW high_risk_companies AS
SELECT * FROM barra_factors
WHERE LEVERAGE > 3 AND VOLTILTY > 50;

-- Anyone can now easily query high-risk companies
SELECT * FROM high_risk_companies WHERE INDNAME1 = 'BANKS';
```

## 3. Subqueries - Inline Nested Queries

### What is a Subquery?
A subquery is a query nested inside another query. It runs first and provides results to the outer query.

### Syntax
```sql
-- In WHERE clause
SELECT * FROM table1
WHERE column > (SELECT AVG(column) FROM table2);

-- In FROM clause
SELECT * FROM (SELECT ... FROM ...) AS derived_table;
```

### When to Use Subqueries
- **Simple one-time calculations** that don't need naming
- **Filtering based on aggregate values** (WHERE clause)
- **Quick comparisons** against calculated values
- **When you only need the result once**

### Example
```sql
-- Find companies with above-average momentum
SELECT NAME, MOMENTUM
FROM barra_factors
WHERE MOMENTUM > (
    SELECT AVG(MOMENTUM)
    FROM barra_factors
    WHERE NONESTU = 0
);
```

## Comparison Table

| Feature | CTE | View | Subquery |
|---------|-----|------|----------|
| **Scope** | Single query | Database-wide | Single query |
| **Persistence** | Temporary | Permanent (until dropped) | Temporary |
| **Reusability** | Within same query | Across all queries | Not reusable |
| **Performance** | Calculated once | Pre-optimized | Calculated each use |
| **Readability** | Excellent | N/A (hidden) | Can be complex |
| **Best for** | Multi-step logic | Common queries | Simple filters |

## Decision Guide

```sql
-- Use CTE when you need intermediate results multiple times in ONE query
WITH industry_avg AS (
    SELECT INDNAME1, AVG(LEVERAGE) as avg_lev
    FROM barra_factors GROUP BY INDNAME1
)
SELECT * FROM barra_factors b
JOIN industry_avg i ON b.INDNAME1 = i.INDNAME1
WHERE b.LEVERAGE > i.avg_lev * 2;

-- Use View when the same logic is needed ACROSS MULTIPLE queries
CREATE VIEW market_statistics AS
SELECT AVG(LEVERAGE) as avg_lev, AVG(MOMENTUM) as avg_mom
FROM barra_factors WHERE NONESTU = 0;

-- Use Subquery for SIMPLE, ONE-TIME calculations
SELECT * FROM barra_factors
WHERE LEVERAGE > (SELECT AVG(LEVERAGE) * 2 FROM barra_factors);
```

## Real-World Analogy
- **CTE**: Like calculating subtotals on scratch paper while solving a math problem
- **View**: Like creating a summary report that many people will reference
- **Subquery**: Like doing a quick mental calculation to make a decision

The key is choosing the right tool for your specific need: CTEs for complex single queries, Views for reusable logic, and Subqueries for simple inline calculations.

In [ ]:
# CTE
# Clear and educational query showing standard deviation calculation
exploratory_stats = """
WITH basic_stats AS (
    SELECT
        AVG(MOMENTUM) as mean,
        AVG(MOMENTUM * MOMENTUM) as mean_of_squares,
        AVG(MOMENTUM) * AVG(MOMENTUM) as square_of_mean,
        AVG(MOMENTUM * MOMENTUM) - (AVG(MOMENTUM) * AVG(MOMENTUM)) as variance,
        SQRT(AVG(MOMENTUM * MOMENTUM) - (AVG(MOMENTUM) * AVG(MOMENTUM))) as my_stddev
    FROM barra_factors
)
SELECT
    mean as "Mean",
    mean_of_squares as "Mean of Squares",
    square_of_mean as "Square of Mean",
    variance as "Variance",
    my_stddev as "Standard Deviation"
FROM basic_stats;
"""

# Execute and show results
pd.read_sql(exploratory_stats, engine)

## Queries to find high momentum stocks with statistical context using all 3 methods.

In [ ]:
# CTE
high_momentum_query = """
WITH momentum_stats AS (
    SELECT
        AVG(MOMENTUM) as mean,
        SQRT(AVG(MOMENTUM * MOMENTUM) - (AVG(MOMENTUM) * AVG(MOMENTUM))) as my_stddev
    FROM barra_factors
    WHERE NONESTU = 0
    AND CAPITALIZATION > 5e9     -- $5 billion threshold

)
SELECT
    TICKER,
    NAME,
    MOMENTUM,
    BETA,
    CAPITALIZATION,
    (SELECT mean FROM momentum_stats) as avg_momentum,
    (SELECT my_stddev FROM momentum_stats) as momentum_stddev,
    (MOMENTUM - (SELECT mean FROM momentum_stats)) /
        (SELECT my_stddev FROM momentum_stats) as z_score
FROM barra_factors
WHERE MOMENTUM > (
    SELECT mean + my_stddev
    FROM momentum_stats
)
AND NONESTU = 0
AND CAPITALIZATION > 5e9     -- $5 billion threshold
ORDER BY MOMENTUM DESC
LIMIT 10;
"""

# Execute and display results
high_momentum_stocks = pd.read_sql(high_momentum_query, engine)
high_momentum_stocks

In [ ]:
from sqlalchemy import text

# View Method
# First create the view (one-time setup)
create_view_query = """
CREATE VIEW IF NOT EXISTS momentum_stats_view AS
SELECT
    AVG(MOMENTUM) as mean,
    SQRT(AVG(MOMENTUM * MOMENTUM) - (AVG(MOMENTUM) * AVG(MOMENTUM))) as my_stddev
FROM barra_factors
WHERE NONESTU = 0
AND CAPITALIZATION > 5e9;
"""

# Create the view
with engine.connect() as conn:
    conn.execute(text(create_view_query))
    conn.commit()

# Now use the view in your query (much simpler!)
high_momentum_view_query = """
SELECT
    TICKER,
    NAME,
    MOMENTUM,
    BETA,
    CAPITALIZATION,
    (SELECT mean FROM momentum_stats_view) as avg_momentum,
    (SELECT my_stddev FROM momentum_stats_view) as momentum_stddev,
    (MOMENTUM - (SELECT mean FROM momentum_stats_view)) /
        (SELECT my_stddev FROM momentum_stats_view) as z_score
FROM barra_factors
WHERE MOMENTUM > (
    SELECT mean + my_stddev
    FROM momentum_stats_view
)
AND NONESTU = 0
AND CAPITALIZATION > 5e9
ORDER BY MOMENTUM DESC
LIMIT 10;
"""

# Execute and display results
high_momentum_stocks_view = pd.read_sql(high_momentum_view_query, engine)
print("Using View:")
high_momentum_stocks_view

# Clean up the view when done
# DROP VIEW IF EXISTS momentum_stats_view;

In [ ]:
# Subquery Method (more complex and potentially less efficient)
high_momentum_subquery = """
SELECT
    TICKER,
    NAME,
    MOMENTUM,
    BETA,
    CAPITALIZATION,
    (SELECT AVG(MOMENTUM)
     FROM barra_factors
     WHERE NONESTU = 0 AND CAPITALIZATION > 5e9) as avg_momentum,
    (SELECT SQRT(AVG(MOMENTUM * MOMENTUM) - (AVG(MOMENTUM) * AVG(MOMENTUM)))
     FROM barra_factors
     WHERE NONESTU = 0 AND CAPITALIZATION > 5e9) as momentum_stddev,
    (MOMENTUM -
        (SELECT AVG(MOMENTUM)
         FROM barra_factors
         WHERE NONESTU = 0 AND CAPITALIZATION > 5e9)) /
        (SELECT SQRT(AVG(MOMENTUM * MOMENTUM) - (AVG(MOMENTUM) * AVG(MOMENTUM)))
         FROM barra_factors
         WHERE NONESTU = 0 AND CAPITALIZATION > 5e9) as z_score
FROM barra_factors
WHERE MOMENTUM > (
    SELECT AVG(MOMENTUM) + SQRT(AVG(MOMENTUM * MOMENTUM) - (AVG(MOMENTUM) * AVG(MOMENTUM)))
    FROM barra_factors
    WHERE NONESTU = 0 AND CAPITALIZATION > 5e9
)
AND NONESTU = 0
AND CAPITALIZATION > 5e9
ORDER BY MOMENTUM DESC
LIMIT 10;
"""

# Execute and display results
high_momentum_stocks_subquery = pd.read_sql(high_momentum_subquery, engine)
print("Using Subqueries:")
print(high_momentum_stocks_subquery)

## Queries to find high beta stocks with statistical context - building block

In [ ]:
# Query to find high beta stocks with statistical context - building block
temp_query = """
    SELECT
        AVG(BETA) as mean,
        SQRT(AVG(BETA * BETA) - (AVG(BETA) * AVG(BETA))) as my_stddev
    FROM barra_factors
    WHERE NONESTU = 0
    AND CAPITALIZATION > 5e9     -- $5 billion threshold
"""

# Execute and display results
pd.read_sql(temp_query, engine)


In [ ]:
# CTE
# Query to find high beta stocks with statistical context
high_beta_query = """
WITH beta_stats AS (
    SELECT
        AVG(BETA) as mean,
        SQRT(AVG(BETA * BETA) - (AVG(BETA) * AVG(BETA))) as my_stddev
    FROM barra_factors
    WHERE NONESTU = 0
    AND CAPITALIZATION > 5e9     -- $5 billion threshold
)
SELECT
    TICKER,
    NAME,
    BETA,
    CAPITALIZATION,
    (SELECT mean FROM beta_stats) as avg_beta,
    (SELECT my_stddev FROM beta_stats) as beta_stddev,
    (BETA - (SELECT mean FROM beta_stats)) /
        (SELECT my_stddev FROM beta_stats) as z_score
FROM barra_factors
WHERE BETA > (
    SELECT mean + my_stddev
    FROM beta_stats
)
AND NONESTU = 0
AND CAPITALIZATION > 5e9     -- $5 billion threshold
ORDER BY BETA DESC
LIMIT 10;
"""

# Execute and display results
high_beta_stocks = pd.read_sql(high_beta_query, engine)
high_beta_stocks

In [ ]:
# CTE method to find stocks with TRISK% > 2.5 std within their own INDNAME1 group
high_trisk_within_industry_query = """
WITH industry_trisk_stats AS (
    -- Calculate TRISK% statistics for each industry
    SELECT
        INDNAME1,
        AVG(`TRISK%`) as industry_mean,
        SQRT(AVG(`TRISK%` * `TRISK%`) - (AVG(`TRISK%`) * AVG(`TRISK%`))) as industry_stddev,
        COUNT(*) as industry_count
    FROM barra_factors
    WHERE NONESTU = 0
    AND CAPITALIZATION > 5e9
    AND INDNAME1 IS NOT NULL
    AND `TRISK%` IS NOT NULL
    GROUP BY INDNAME1
),
stocks_with_z_scores AS (
    -- Calculate z-scores for all stocks
    SELECT
        b.INDNAME1,
        b.TICKER,
        b.NAME,
        b.`TRISK%`,
        b.MOMENTUM,
        b.BETA,
        b.CAPITALIZATION,
        s.industry_mean,
        s.industry_stddev,
        (b.`TRISK%` - s.industry_mean) / s.industry_stddev as industry_z_score,
        s.industry_count
    FROM barra_factors b
    INNER JOIN industry_trisk_stats s ON b.INDNAME1 = s.INDNAME1
    WHERE b.NONESTU = 0
    AND b.CAPITALIZATION > 5e9
    AND b.`TRISK%` IS NOT NULL
    AND (b.`TRISK%` - s.industry_mean) / s.industry_stddev > 2.5  -- More than 2.5 std above industry mean
),
industry_summary AS (
    -- Create summary statistics by industry
    SELECT
        INDNAME1,
        COUNT(*) as extreme_stock_count,
        MIN(industry_z_score) as min_z_score,
        MAX(industry_z_score) as max_z_score,
        AVG(industry_z_score) as avg_z_score,
        MIN(`TRISK%`) as min_trisk,
        MAX(`TRISK%`) as max_trisk,
        AVG(`TRISK%`) as avg_trisk
    FROM stocks_with_z_scores
    GROUP BY INDNAME1
)
-- Main query showing individual stocks
SELECT * FROM stocks_with_z_scores
ORDER BY INDNAME1, industry_z_score DESC;
"""

# Execute and display results
extreme_trisk_stocks = pd.read_sql(high_trisk_within_industry_query, engine)
print(f"Stocks with TRISK% > 2.5 standard deviations above their industry mean:")
print(f"Total stocks found: {len(extreme_trisk_stocks)}")
print("\n", extreme_trisk_stocks)

# Separate query for industry summary
industry_summary_query = """
WITH industry_trisk_stats AS (
    SELECT
        INDNAME1,
        AVG(`TRISK%`) as industry_mean,
        SQRT(AVG(`TRISK%` * `TRISK%`) - (AVG(`TRISK%`) * AVG(`TRISK%`))) as industry_stddev,
        COUNT(*) as industry_count
    FROM barra_factors
    WHERE NONESTU = 0
    AND CAPITALIZATION > 5e9
    AND INDNAME1 IS NOT NULL
    AND `TRISK%` IS NOT NULL
    GROUP BY INDNAME1
),
stocks_with_z_scores AS (
    SELECT
        b.INDNAME1,
        b.TICKER,
        b.`TRISK%`,
        (b.`TRISK%` - s.industry_mean) / s.industry_stddev as industry_z_score
    FROM barra_factors b
    INNER JOIN industry_trisk_stats s ON b.INDNAME1 = s.INDNAME1
    WHERE b.NONESTU = 0
    AND b.CAPITALIZATION > 5e9
    AND b.`TRISK%` IS NOT NULL
    AND (b.`TRISK%` - s.industry_mean) / s.industry_stddev > 2.5
)
SELECT
    INDNAME1,
    COUNT(*) as extreme_stock_count,
    ROUND(MIN(industry_z_score), 2) as min_z_score,
    ROUND(MAX(industry_z_score), 2) as max_z_score,
    ROUND(AVG(industry_z_score), 2) as avg_z_score,
    ROUND(MIN(`TRISK%`), 2) as min_trisk_pct,
    ROUND(MAX(`TRISK%`), 2) as max_trisk_pct,
    ROUND(AVG(`TRISK%`), 2) as avg_trisk_pct
FROM stocks_with_z_scores
GROUP BY INDNAME1
ORDER BY extreme_stock_count DESC, max_z_score DESC;
"""

print("\n\nSummary by Industry (all grouping done in SQL):")
industry_summary = pd.read_sql(industry_summary_query, engine)
print(industry_summary)

In [ ]:
# View Method
# Create view for beta statistics (one-time setup)
create_beta_view_query = """
CREATE VIEW IF NOT EXISTS beta_stats_view AS
SELECT
    AVG(BETA) as mean,
    SQRT(AVG(BETA * BETA) - (AVG(BETA) * AVG(BETA))) as my_stddev
FROM barra_factors
WHERE NONESTU = 0
AND CAPITALIZATION > 5e9;
"""

# Create the view
with engine.connect() as conn:
    conn.execute(text(create_beta_view_query))
    conn.commit()

# Use the view in your query
high_beta_view_query = """
SELECT
    TICKER,
    NAME,
    BETA,
    CAPITALIZATION,
    (SELECT mean FROM beta_stats_view) as avg_beta,
    (SELECT my_stddev FROM beta_stats_view) as beta_stddev,
    (BETA - (SELECT mean FROM beta_stats_view)) /
        (SELECT my_stddev FROM beta_stats_view) as z_score
FROM barra_factors
WHERE BETA > (
    SELECT mean + my_stddev
    FROM beta_stats_view
)
AND NONESTU = 0
AND CAPITALIZATION > 5e9
ORDER BY BETA DESC
LIMIT 10;
"""

# Execute and display results
high_beta_stocks_view = pd.read_sql(high_beta_view_query, engine)
print("Using View:")
print(high_beta_stocks_view)

# Clean up: DROP VIEW IF EXISTS beta_stats_view;

In [ ]:
# Pure Subquery Method (repetitive and less efficient)
high_beta_subquery = """
SELECT
    TICKER,
    NAME,
    BETA,
    CAPITALIZATION,
    (SELECT AVG(BETA)
     FROM barra_factors
     WHERE NONESTU = 0 AND CAPITALIZATION > 5e9) as avg_beta,
    (SELECT SQRT(AVG(BETA * BETA) - (AVG(BETA) * AVG(BETA)))
     FROM barra_factors
     WHERE NONESTU = 0 AND CAPITALIZATION > 5e9) as beta_stddev,
    (BETA -
        (SELECT AVG(BETA)
         FROM barra_factors
         WHERE NONESTU = 0 AND CAPITALIZATION > 5e9)) /
        (SELECT SQRT(AVG(BETA * BETA) - (AVG(BETA) * AVG(BETA)))
         FROM barra_factors
         WHERE NONESTU = 0 AND CAPITALIZATION > 5e9) as z_score
FROM barra_factors
WHERE BETA > (
    SELECT AVG(BETA) + SQRT(AVG(BETA * BETA) - (AVG(BETA) * AVG(BETA)))
    FROM barra_factors
    WHERE NONESTU = 0 AND CAPITALIZATION > 5e9
)
AND NONESTU = 0
AND CAPITALIZATION > 5e9
ORDER BY BETA DESC
LIMIT 10;
"""

# Execute and display results
high_beta_stocks_subquery = pd.read_sql(high_beta_subquery, engine)
print("Using Pure Subqueries:")
print(high_beta_stocks_subquery)

In [ ]:
# Compare all methods
import time

methods = {
    "CTE (Original)": high_beta_query,
    "View": high_beta_view_query,  # Assumes view exists
    "Pure Subquery": high_beta_subquery,
}

print("\n=== Performance Comparison ===")
for name, query in methods.items():
    start = time.time()
    result = pd.read_sql(query, engine)
    elapsed = time.time() - start
    print(f"{name:20} : {elapsed:.4f} seconds")

# Verify all methods return same results
print("\n=== Results Verification ===")
print(f"All methods return same results: {len(set([len(df) for df in [high_beta_stocks, high_beta_stocks_view, high_beta_stocks_subquery, ]])) == 1}")

Turns out, the type of database matters...

* "MATERIALIZED" in database terms means physically storing the results rather than just storing the query definition. Let me explain with clear examples:



In [ ]:
performance_notes = """
Different databases handle these differently:

SQLite (what you're using):
- CTEs: Well optimized, materialized once
- Views: Not materialized, recalculated each access
- Subqueries: Some caching, but still repetitive

PostgreSQL:
- CTEs: Can be MATERIALIZED or NOT MATERIALIZED (you choose)
- Views: Can create MATERIALIZED VIEWs for better performance
- Subqueries: Sophisticated optimizer, often rewrites them

MySQL:
- CTEs: Materialized by default (good for this case)
- Views: Two algorithms (MERGE or TEMPTABLE)
- Subqueries: Historically poor, much better in recent versions

SQL Server:
- CTEs: Not always materialized (optimizer decides)
- Views: Can create INDEXED views (materialized)
- Subqueries: Generally well optimized
"""
print(performance_notes)

# Showing Database Schema
We can examine the structure of our Barra database:

In [ ]:
# Show table structure
schema_query = """
SELECT
    name, type
FROM sqlite_master
WHERE type='table';
"""

# Show columns and their types
column_query = """
PRAGMA table_info(barra_factors);
"""

print("Tables in Database:")
print(pd.read_sql(schema_query, engine))
print("\nColumns in barra_factors table:")
print(pd.read_sql(column_query, engine))

# Understanding Database Indexes in SQL

## What is an Index?
Think of an index in a database like an index in a book:
- A book index helps you find pages without reading the whole book
- A database index helps find data without scanning the whole table
- Both trade storage space for faster lookup speed

## Types of Indexes

### 1. Single-Column Index
```sql
CREATE INDEX idx_barrid ON barra_factors(BARRID);
```
- Like looking up words in a book's index
- Best for finding exact matches
- Example: Finding a specific stock by its BARRID

### 2. Composite Index (Multiple Columns)
```sql
CREATE INDEX idx_ind_cap ON barra_factors(INDNAME1, CAPITALIZATION);
```
- Like a book indexed by both topic and subtopic
- Useful for queries that filter on multiple columns
- Example: Finding banks with market cap > $1B

## When Indexes Help

### Good Use Cases:
1. **WHERE Clauses**
   ```sql
   WHERE BARRID = 'ABC123'  -- Very fast with index
   ```

2. **JOIN Operations**
   ```sql
   JOIN another_table ON barra_factors.BARRID = another_table.BARRID
   ```

3. **ORDER BY**
   ```sql
   ORDER BY CAPITALIZATION DESC  -- Faster with index
   ```

### When Indexes Don't Help:
1. **Function on Column**
   ```sql
   WHERE LOWER(INDNAME1) = 'banks'  -- Index not used
   ```

2. **Leading Wildcard**
   ```sql
   WHERE BARRID LIKE '%123'  -- Index not used
   ```

## Performance Trade-offs

### Advantages:
- Faster SELECT queries
- Faster WHERE clause evaluation
- Improved sorting performance

### Disadvantages:
- Extra disk space needed
- Slower INSERT and UPDATE operations
- Maintenance overhead

## Real-World Analogy
Think of a library:
- Books without index: Must scan every page to find content
- Books with index: Jump directly to relevant pages
- Card catalog: Like a database index for all books


In [ ]:
from sqlalchemy import text
import time

# Create copies of the table - one with index, one without
create_queries = [
    text("CREATE TABLE barra_factors_no_index AS SELECT * FROM barra_factors;"),
    text("CREATE TABLE barra_factors_with_index AS SELECT * FROM barra_factors;"),
    text("CREATE TABLE barra_factors_with_bad_index AS SELECT * FROM barra_factors;"),

    text("CREATE INDEX idx_barrid ON barra_factors_with_index(BARRID);"),
    text("""
    -- Composite Indexes: Multiple columns in one index
    -- • Order of columns matters (most selective first)
    -- • Can speed up queries filtering/sorting on multiple columns
    -- • Takes more space than single-column indexes
    -- Create composite index for common query patterns
    CREATE INDEX idx_industry_leverage
    ON barra_factors_with_index(INDNAME1, LEVERAGE DESC);
    """),

    text("""
    -- Create BAD composite index for common query patterns
    CREATE INDEX idx_leverage_industry
    ON barra_factors_with_bad_index(LEVERAGE DESC, INDNAME1);
    """),


]

# Execute the queries
with engine.connect() as conn:
    for query in create_queries:
      try:
        conn.execute(query)
        conn.commit()  # Need to commit the changes
      except:
        print(f"Query failed: {query}")
        continue


In [ ]:
# Query to show all indices in the database
index_query = text("""
SELECT
    type,
    name,
    tbl_name,
    sql
FROM sqlite_master
WHERE type = 'index';
""")

# Execute and display results
with engine.connect() as conn:
    indices = pd.read_sql(index_query, engine)
print("\nIndices in database:")
print(indices)

In [ ]:
# Get a sample BARRID for testing
sample_barrid_query = text("SELECT BARRID FROM barra_factors LIMIT 1")
with engine.connect() as conn:
    test_barrid = conn.execute(sample_barrid_query).fetchone()[0]
print(f"Sample BARRID: {test_barrid}")

# Test queries
test_queries = {
    'Without Index': text(f"SELECT * FROM barra_factors_no_index WHERE BARRID = '{test_barrid}'"),
    'With Index': text(f"SELECT * FROM barra_factors_with_index WHERE BARRID = '{test_barrid}'")
}

# Time the queries
results = {}
for name, query in test_queries.items():
    start_time = time.time()
    with engine.connect() as conn:
        result = conn.execute(query)
        _ = result.fetchall()
    end_time = time.time()
    results[name] = end_time - start_time

# Print results
print(f"Query time without index: {results['Without Index']:.4f} seconds")
print(f"Query time with index: {results['With Index']:.4f} seconds")
print(f"Speed improvement: {results['Without Index']/results['With Index']:.1f}x faster")

In [ ]:
# Test queries
test_queries = {
    # 'Without Index': text("SELECT * FROM barra_factors_no_index WHERE INDNAME1 in ('BANKS','SECASSET','THRIFTS','LIFEINS') ORDER BY LEVERAGE DESC;"),
    # 'With Index': text("SELECT * FROM barra_factors_with_index WHERE INDNAME1 in ('BANKS','SECASSET','THRIFTS','LIFEINS')  ORDER BY LEVERAGE DESC;"),
    # 'With BAD Index': text("SELECT * FROM barra_factors_with_bad_index WHERE INDNAME1 in ('BANKS','SECASSET','THRIFTS','LIFEINS')  ORDER BY LEVERAGE DESC;")

    'Without Index': text("SELECT * FROM barra_factors_no_index WHERE INDNAME1 in ('BANKS','SECASSET','THRIFTS','LIFEINS') ORDER BY LEVERAGE DESC;"),
    'With Index': text("SELECT * FROM barra_factors_with_index WHERE INDNAME1 in ('BANKS','SECASSET','THRIFTS','LIFEINS')  ORDER BY LEVERAGE DESC;"),
    'With BAD Index': text("SELECT * FROM barra_factors_with_bad_index WHERE INDNAME1 in ('BANKS','SECASSET','THRIFTS','LIFEINS')  ORDER BY LEVERAGE DESC;")


    }

# Time the queries
results = {}
for loop in range(100):
  for name, query in test_queries.items():
      start_time = time.time()
      with engine.connect() as conn:
          result = conn.execute(query)
          _ = result.fetchall()
      end_time = time.time()
      if not name in results:
        results[name] =  end_time - start_time
      else:
        results[name] +=  end_time - start_time


# Print results
print(f"Query time without index: {results['Without Index']:.4f} seconds")
print(f"Query time with index: {results['With Index']:.4f} seconds")
print(f"Query time with BAD index: {results['With BAD Index']:.4f} seconds")
print(f"Speed improvement: {results['Without Index']/results['With Index']:.1f}x faster")

# EXPLAIN Plans
- Understanding query execution plans
- EXPLAIN shows how SQLite will execute your query (which indexes, scan types, etc.)
- Helps identify performance bottlenecks and missing indexes
- Different databases have different EXPLAIN syntax (EXPLAIN ANALYZE in PostgreSQL)



In [ ]:
query = text("""
EXPLAIN QUERY PLAN
SELECT * FROM barra_factors_no_index WHERE BARRID = 'USA1131';
""")

# Execute and display results
with engine.connect() as conn:
    print(pd.read_sql(query, engine).T.values) # SCAN - full table scan


In [ ]:
query = text("""
EXPLAIN QUERY PLAN
SELECT * FROM barra_factors_with_index WHERE BARRID = 'USA1131';
""")

# Execute and display results
with engine.connect() as conn:
    print(pd.read_sql(query, engine).T.values) # SEARCH -- using index


# Database Transactions: Ensuring Data Integrity

## What is a Transaction?
A transaction is a sequence of database operations that are treated as a single unit of work. Either ALL operations succeed, or NONE of them do. This "all-or-nothing" approach ensures your database never ends up in an inconsistent state.

## The ACID Properties
Transactions guarantee four critical properties:

### **A**tomicity
- All operations in a transaction succeed or all fail
- No partial updates allowed
- Like a bank transfer: money leaves one account AND enters another, or neither happens

### **C**onsistency
- Database remains in a valid state before and after the transaction
- All rules and constraints are maintained
- Prevents corruption of data relationships

### **I**solation
- Concurrent transactions don't interfere with each other
- Each transaction operates as if it's the only one running
- Prevents "dirty reads" of uncommitted data

### **D**urability
- Once committed, changes persist even if system crashes
- Data is written to permanent storage
- Survived power outages and system failures

## Basic Transaction Syntax

```sql
-- Start a transaction
BEGIN TRANSACTION;  -- or just BEGIN

-- Your SQL operations here
UPDATE accounts SET balance = balance - 100 WHERE id = 1;
UPDATE accounts SET balance = balance + 100 WHERE id = 2;

-- End the transaction (choose one):
COMMIT;    -- Save all changes permanently
ROLLBACK;  -- Undo all changes since BEGIN
```

## Real-World Example: Bank Transfer

```sql
-- Scenario: Transfer $1000 from Alice to Bob
BEGIN TRANSACTION;

-- Deduct from Alice
UPDATE accounts
SET balance = balance - 1000
WHERE customer_name = 'Alice';

-- Check if Alice had sufficient funds
SELECT balance FROM accounts WHERE customer_name = 'Alice';
-- If balance is negative, we should ROLLBACK

-- Add to Bob
UPDATE accounts
SET balance = balance + 1000
WHERE customer_name = 'Bob';

-- If everything looks good:
COMMIT;
-- If there's any problem:
-- ROLLBACK;
```

## Transaction in Your Barra Context

```sql
-- Safe way to adjust leverage for risk analysis
BEGIN TRANSACTION;

-- Save original state
CREATE TEMP TABLE leverage_backup AS
SELECT BARRID, LEVERAGE FROM barra_factors WHERE INDNAME1 = 'BANKS';

-- Simulate leverage increase
UPDATE barra_factors
SET LEVERAGE = LEVERAGE * 1.1
WHERE INDNAME1 = 'BANKS';

-- Analyze the impact
SELECT
    COUNT(*) as high_risk_count,
    AVG(LEVERAGE) as new_avg_leverage
FROM barra_factors
WHERE INDNAME1 = 'BANKS' AND LEVERAGE > 3;

-- Decide based on analysis:
-- ROLLBACK;  -- Undo the simulation
-- or
-- COMMIT;    -- Keep the changes
```

## SQLite Transaction Modes

```sql
-- SQLite supports three transaction modes:

-- 1. DEFERRED (default)
BEGIN DEFERRED TRANSACTION;
-- Acquires locks only when needed
-- Allows concurrent reads

-- 2. IMMEDIATE
BEGIN IMMEDIATE TRANSACTION;
-- Acquires write lock immediately
-- Prevents other writes but allows reads

-- 3. EXCLUSIVE
BEGIN EXCLUSIVE TRANSACTION;
-- Acquires exclusive lock
-- No other connections can read or write
```

## Common Use Cases

### 1. **Data Import**
```sql
BEGIN TRANSACTION;
-- Import thousands of records
INSERT INTO barra_factors VALUES (...);
INSERT INTO barra_factors VALUES (...);
-- ... many more inserts
COMMIT;  -- All succeed or all fail
```

### 2. **Cleanup Operations**
```sql
BEGIN TRANSACTION;
-- Delete related data from multiple tables
DELETE FROM order_items WHERE order_id = 123;
DELETE FROM orders WHERE id = 123;
DELETE FROM customer_history WHERE order_id = 123;
COMMIT;  -- Ensures referential integrity
```

### 3. **Complex Updates**
```sql
BEGIN TRANSACTION;
-- Rebalance portfolio
UPDATE holdings SET weight = weight * 0.9 WHERE sector = 'TECH';
UPDATE holdings SET weight = weight * 1.1 WHERE sector = 'FINANCE';
-- Verify weights still sum to 100%
SELECT SUM(weight) FROM holdings;
-- If correct: COMMIT;
-- If wrong: ROLLBACK;
```

## Automatic Transactions in SQLite

```sql
-- SQLite wraps individual statements in implicit transactions
INSERT INTO test VALUES (1);  -- Automatically wrapped in BEGIN...COMMIT

-- But for multiple statements, explicit is better:
BEGIN;
INSERT INTO test VALUES (1);
INSERT INTO test VALUES (2);
COMMIT;  -- Both inserts succeed or both fail
```

## Error Handling Pattern

```python
# Python example with error handling
import sqlite3

conn = sqlite3.connect('database.db')
try:
    conn.execute("BEGIN")
    conn.execute("UPDATE accounts SET balance = balance - 100 WHERE id = 1")
    conn.execute("UPDATE accounts SET balance = balance + 100 WHERE id = 2")
    
    # Check for business logic
    result = conn.execute("SELECT balance FROM accounts WHERE id = 1").fetchone()
    if result[0] < 0:
        raise ValueError("Insufficient funds")
    
    conn.commit()
    print("Transaction successful")
except Exception as e:
    conn.rollback()
    print(f"Transaction failed: {e}")
finally:
    conn.close()
```

## Key Points to Remember

1. **Transactions are temporary** until committed
2. **ROLLBACK undoes everything** since BEGIN
3. **Nested transactions** aren't supported in SQLite
4. **Automatic rollback** occurs if connection closes without commit
5. **Write operations lock the database** in SQLite (it's single-writer)

The power of transactions is that they protect your data integrity even when things go wrong - network failures, power outages, or logical errors in your code. They ensure your database is always in a consistent, valid state!

In [ ]:
# Example 1: Stress test simulation with rollback

with engine.connect() as conn:
    # Start transaction
    trans = conn.begin()

    try:
        # Create backup table
        conn.execute(text("""
            CREATE TEMP TABLE IF NOT EXISTS leverage_backup AS
            SELECT BARRID, LEVERAGE, CAPITALIZATION
            FROM barra_factors
            WHERE INDNAME1 IN ('BANKS', 'THRIFTS')
        """))

        # Get before state
        before = pd.read_sql(text("""
            SELECT COUNT(*) as High_Risk_Count
            FROM leverage_backup
            WHERE LEVERAGE > 3
        """), conn)

        # Simulate stress scenario
        conn.execute(text("""
            UPDATE barra_factors
            SET LEVERAGE = LEVERAGE * 1.5
            WHERE INDNAME1 IN ('BANKS', 'THRIFTS')
            AND NONESTU = 0
        """))

        # Get after state
        after = pd.read_sql(text("""
            SELECT COUNT(*) as High_Risk_Count
            FROM barra_factors
            WHERE INDNAME1 IN ('BANKS', 'THRIFTS')
            AND LEVERAGE > 3
        """), conn)

        # Show results
        print("Stress Test Results:")
        print(f"Before: {before['High_Risk_Count'][0]} high-risk companies")
        print(f"After:  {after['High_Risk_Count'][0]} high-risk companies")

        # Always rollback for simulations
        trans.rollback()
        print("\nChanges rolled back - database unchanged")

    except Exception as e:
        trans.rollback()
        print(f"Error occurred: {e}")
        raise

In [ ]:
import sqlite3
import pandas as pd

db_path = engine.url.database
conn = sqlite3.connect(db_path)

try:
    # Start transaction
    conn.execute("BEGIN TRANSACTION")

    # Create backup focusing on Lehman
    conn.execute("""
        CREATE TEMP TABLE IF NOT EXISTS test_backup AS
        SELECT * FROM barra_factors
        WHERE BARRID = 'USARHS2'
        ORDER BY LEVERAGE DESC
        LIMIT 10
    """)

    # Get BEFORE state using pandas
    before_df = pd.read_sql("""
        SELECT NAME, LEVERAGE, CAPITALIZATION/1e9 as CAP_B
        FROM test_backup
        ORDER BY LEVERAGE DESC
    """, conn)

    print("BEFORE Update (Top 10 banks by leverage):")
    print(before_df.to_string(index=False))

    # Update
    conn.execute("UPDATE test_backup SET LEVERAGE = LEVERAGE * 2")

    # Get AFTER state
    after_df = pd.read_sql("""
        SELECT NAME, LEVERAGE, CAPITALIZATION/1e9 as CAP_B
        FROM test_backup
        ORDER BY LEVERAGE DESC
    """, conn)

    print("\nAFTER Update (Leverage x2):")
    print(after_df.to_string(index=False))

    # Show Lehman specifically
    lehman_before = before_df[before_df['NAME'].str.contains('LEHMAN')]
    lehman_after = after_df[after_df['NAME'].str.contains('LEHMAN')]

    if not lehman_before.empty:
        print(f"\nLehman Brothers specifically:")
        print(f"  Before: {lehman_before.iloc[0]['LEVERAGE']:.3f}")
        print(f"  After:  {lehman_after.iloc[0]['LEVERAGE']:.3f}")

    # Rollback
    conn.execute("ROLLBACK")
    print("\n✓ Transaction rolled back")

except Exception as e:
    print(f"Error: {e}")
    conn.rollback()
finally:
    conn.close()

# Understanding Primary Keys in SQL

## What is a Primary Key?
A primary key is a column (or set of columns) that:
- Uniquely identifies each row in a table
- Cannot contain NULL values
- Must have unique values
- Automatically creates an index
- Only one primary key per table

## Real World Analogy
Think of:
- Social Security Numbers for people
- ISBN numbers for books
- License plate numbers for cars
- Student ID numbers in a university

## Creating Primary Keys

### During Table Creation
```sql
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,  -- Single column primary key
    name TEXT,
    grade REAL
);
```

### Composite Primary Key (Multiple Columns)
```sql
CREATE TABLE class_enrollment (
    student_id INTEGER,
    course_id INTEGER,
    semester TEXT,
    PRIMARY KEY (student_id, course_id, semester)
);
```

## Benefits of Primary Keys

1. **Data Integrity**
   - Prevents duplicate records
   - Ensures each record can be uniquely identified
   - No NULL values allowed

2. **Referential Integrity**
   - Other tables can reference this unique identifier
   - Enables proper relationships between tables
   - Foundation for foreign keys

3. **Performance**
   - Automatically creates an index
   - Speeds up searches and joins
   - Optimizes database operations

## Primary Key vs. Index
| Feature | Primary Key | Index |
|---------|-------------|--------|
| Uniqueness | Required | Optional |
| NULL values | Not allowed | Allowed |
| Per table | Only one | Multiple allowed |
| Purpose | Data integrity | Performance |
| Auto-indexed | Yes | N/A |

## Best Practices
1. Choose meaningful primary keys when possible
2. Use simple and stable values
3. Avoid using data that might change
4. Consider using auto-incrementing numbers for simplicity
5. Keep the primary key as small as possible


In [ ]:
# First, let's check if BARRID is unique
check_unique = text("""
SELECT BARRID, COUNT(*) as count
FROM barra_factors
GROUP BY BARRID
HAVING count > 1;
""")

# Execute check
with engine.connect() as conn:
    duplicates = pd.read_sql(check_unique, engine)
print("Checking for duplicate BARRIDs:")
print(duplicates)

In [ ]:
# Create new table with primary key
create_pk_table = text("""
CREATE TABLE barra_factors_pk (
    BARRID TEXT PRIMARY KEY,
    TICKER TEXT,
    CUSIP TEXT,
    NAME TEXT,
    HBTA REAL,
    BETA REAL,
    "SRISK%" REAL,
    "TRISK%" REAL,
    VOLTILTY REAL,
    MOMENTUM REAL,
    SIZE REAL,
    SIZENONL REAL,
    TRADEACT REAL,
    GROWTH REAL,
    EARNYLD REAL,
    VALUE REAL,
    EARNVAR REAL,
    LEVERAGE REAL,
    CURRSEN REAL,
    YIELD REAL,
    NONESTU REAL,
    INDNAME1 TEXT,
    IND1 INTEGER,
    "WGT1%" INTEGER,
    INDNAME2 TEXT,
    IND2 INTEGER,
    "WGT2%" INTEGER,
    INDNAME3 TEXT,
    IND3 INTEGER,
    "WGT3%" INTEGER,
    INDNAME4 TEXT,
    IND4 INTEGER,
    "WGT4%" INTEGER,
    INDNAME5 TEXT,
    IND5 INTEGER,
    "WGT5%" INTEGER,
    PRICE REAL,
    CAPITALIZATION REAL,
    "YLD%" REAL,
    SAP500 INTEGER,
    SAPVAL INTEGER,
    SAPGRO INTEGER,
    MIDCAP INTEGER,
    MIDVAL INTEGER,
    MIDGRO INTEGER,
    SC600 INTEGER,
    SCVAL INTEGER,
    SCGRO INTEGER,
    E3ESTU INTEGER
);
""")

# Copy data to new table
copy_data = text("""
INSERT INTO barra_factors_pk
SELECT * FROM barra_factors;
""")

# Execute creation and copy
with engine.connect() as conn:
    conn.execute(create_pk_table)
    conn.execute(copy_data)
    conn.commit()

In [ ]:
# Verify primary key
verify_pk = text("""
SELECT * FROM pragma_table_info('barra_factors_pk')
""")

with engine.connect() as conn:
    pk_info = pd.read_sql(verify_pk, engine)
print("\nPrimary Key Information:")
print(pk_info)

In [ ]:
# Get a sample BARRID for testing
sample_barrid_query = text("SELECT BARRID FROM barra_factors LIMIT 1")
with engine.connect() as conn:
    test_barrid = conn.execute(sample_barrid_query).fetchone()[0]
print(f"Sample BARRID: {test_barrid}")

# Test queries
test_queries = {
    'Without Index': text(f"SELECT * FROM barra_factors_no_index WHERE BARRID = '{test_barrid}'"),
    'With Index': text(f"SELECT * FROM barra_factors_with_index WHERE BARRID = '{test_barrid}'"),
    'With PK': text(f"SELECT * FROM barra_factors_pk WHERE BARRID = '{test_barrid}'")
}

# Time the queries
results = {}
for name, query in test_queries.items():
    start_time = time.time()
    with engine.connect() as conn:
        result = conn.execute(query)
        _ = result.fetchall()
    end_time = time.time()
    results[name] = end_time - start_time

# Print results
print(f"Query time without index: {results['Without Index']:.4f} seconds")
print(f"Query time with index: {results['With Index']:.4f} seconds")
print(f"Query time with PK: {results['With PK']:.4f} seconds")


```-- Key Differences Summary:
/*
┌─────────────────┬──────────────────┬─────────────────┐
│ Feature         │ Primary Key      │ Index           │
├─────────────────┼──────────────────┼─────────────────┤
│ Purpose         │ Identify rows    │ Speed searches  │
│ Number allowed  │ ONE per table    │ MANY per table  │
│ Uniqueness      │ REQUIRED         │ Optional        │
│ NULL values     │ NOT allowed      │ Allowed         │
│ Auto-indexed?   │ YES              │ IS the index    │
│ Business rule?  │ YES              │ No              │
└─────────────────┴──────────────────┴─────────────────┘
*/
```

- Think of it this way:
- Primary Key = Social Security Number (unique ID for each person, business defines this as the unique identifier)
- Index = Phone book listings (organized for fast lookup, many people can share last names)


# Understanding Performance: Primary Keys vs. Indices

## Key Performance Comparison
Based on the sample results:
```
Query time without index: 0.0038 seconds
Query time with index:    0.0004 seconds
Query time with PK:       0.0003 seconds
```

## Why PK is (Usually) Faster
1. **Automatic Clustering**
   - PKs often physically organize data
   - Database typically clusters data around PK
   - Direct access to disk location

2. **Unique Index**
   - PK automatically creates a unique index
   - Can stop searching after first match
   - No need to check for duplicates

## However, PKs Aren't Always Faster
PKs might be slower when:
1. **Composite Keys**
   ```sql
   PRIMARY KEY (BARRID, date)  -- Multiple columns
   ```
   vs.
   ```sql
   INDEX idx_barrid (BARRID)   -- Single column
   ```

2. **Large Primary Keys**
   - If PK is large (like a long string)
   - Takes more space in index
   - More I/O operations needed

3. **Non-Sequential Values**
   - Random strings (like BARRIDs)
   - vs. Auto-incrementing integers
   - Can cause page splits

## Best Practice
- Use PK for data integrity
- Add separate indices for performance if needed
- Monitor query performance
- Choose based on your specific use case


# Which companies are most affected by the Global Financial Crisis?
* So far data BEFORE the crisis 12/2007
* We need data AFTER the crash 12/2008

In [ ]:
barra_df_2008 = pd.read_csv('USE3L0812.RSK',skiprows=1)
barra_df_2008.columns=barra_df_2008.columns.str.strip()
for col in barra_df_2008.select_dtypes(include='object').columns:
  barra_df_2008[col] = barra_df_2008[col].str.strip()

In [ ]:
try:
  with engine.connect() as conn:
      conn.execute(text("DROP TABLE barra_2008_pk;"))
except:
  0

# Create table with BARRID as primary key directly from DataFrame
create_pk_table = text("""
CREATE TABLE barra_2008_pk (
    BARRID TEXT PRIMARY KEY,
    TICKER TEXT,
    CUSIP TEXT,
    NAME TEXT,
    HBTA REAL,
    BETA REAL,
    "SRISK%" REAL,
    "TRISK%" REAL,
    VOLTILTY REAL,
    MOMENTUM REAL,
    SIZE REAL,
    SIZENONL REAL,
    TRADEACT REAL,
    GROWTH REAL,
    EARNYLD REAL,
    VALUE REAL,
    EARNVAR REAL,
    LEVERAGE REAL,
    CURRSEN REAL,
    YIELD REAL,
    NONESTU REAL,
    INDNAME1 TEXT,
    IND1 INTEGER,
    "WGT1%" INTEGER,
    INDNAME2 TEXT,
    IND2 INTEGER,
    "WGT2%" INTEGER,
    INDNAME3 TEXT,
    IND3 INTEGER,
    "WGT3%" INTEGER,
    INDNAME4 TEXT,
    IND4 INTEGER,
    "WGT4%" INTEGER,
    INDNAME5 TEXT,
    IND5 INTEGER,
    "WGT5%" INTEGER,
    PRICE REAL,
    CAPITALIZATION REAL,
    "YLD%" REAL,
    SAP500 INTEGER,
    SAPVAL INTEGER,
    SAPGRO INTEGER,
    MIDCAP INTEGER,
    MIDVAL INTEGER,
    MIDGRO INTEGER,
    SC600 INTEGER,
    SCVAL INTEGER,
    SCGRO INTEGER,
    E3ESTU INTEGER
);
""")

# Execute table creation and insert data
with engine.connect() as conn:
    conn.execute(create_pk_table)


In [ ]:

# Create table and load data
barra_df_2008.to_sql('barra_2008_pk',
                     engine,
                     if_exists='append', #not replace!
                     index=False,
                     dtype=dtype_dict)


In [ ]:
# Verify primary key
verify_pk = text("""
SELECT * FROM pragma_table_info('barra_2008_pk')
""")

with engine.connect() as conn:
    pk_info = pd.read_sql(verify_pk, engine)
print("\nPrimary Key Information:")
print(pk_info)


# Understanding SQL JOINs and Table Aliases

## What is a JOIN?
A JOIN combines rows from two or more tables based on a related column between them. Think of it like:
- Matching records in an Excel VLOOKUP
- Connecting puzzle pieces that fit together
- Merging two spreadsheets based on a common ID

## Table Aliases (a and b)
In our query:
```sql
FROM barra_factors a       -- 'a' is 2007 data
JOIN barra_2008_pk b      -- 'b' is 2008 data
```
- `a` is a nickname for the 2007 table (barra_factors)
- `b` is a nickname for the 2008 table (barra_2008_pk)
- Makes the code shorter and clearer
- Helps distinguish which table each column comes from

## Breaking Down the Query

```sql
SELECT
    a.BARRID,             -- BARRID from 2007 table
    a.NAME,               -- Company name from 2007 table
    a.INDNAME1,           -- Industry from 2007 table
    ROUND(a.CAPITALIZATION/1e9, 2) as CAP_2007_B,    -- 2007 market cap
    ROUND(b.CAPITALIZATION/1e9, 2) as CAP_2008_B,    -- 2008 market cap
    ROUND((b.CAPITALIZATION - a.CAPITALIZATION)/a.CAPITALIZATION * 100, 2) as PCT_CHANGE  -- % change
```

## How the JOIN Works
```sql
FROM barra_factors a
JOIN barra_2008_pk b ON a.BARRID = b.BARRID
```
- Matches rows where BARRID is the same in both tables
- Like this:
```
2007 (a)    JOIN   2008 (b)
BARRID=ABC  ←→     BARRID=ABC
BARRID=XYZ  ←→     BARRID=XYZ
```

## Filters in the Query
```sql
WHERE a.NONESTU = 0    -- Only estimation universe stocks
    AND a.CAPITALIZATION > 1e9           -- Only billion+ companies
```

## Visual Example:
```
2007 Table (a)          2008 Table (b)           RESULT
BARRID  CAP             BARRID  CAP      →       BARRID  CAP_2007  CAP_2008
ABC     100             ABC     90        →       ABC     100       90
XYZ     200             XYZ     150       →       XYZ     200       150
DEF     300             MNO     175       →       DEF     300       [no match]
```


## INNER JOIN

This is an INNER JOIN (when you write just JOIN, it defaults to INNER JOIN).



In [ ]:

# Now we can run our comparison query
comparison_query = text("""
SELECT
    a.BARRID,
    a.NAME,
    a.INDNAME1,
    ROUND(a.CAPITALIZATION/1e9, 2) as CAP_2007_B,
    ROUND(b.CAPITALIZATION/1e9, 2) as CAP_2008_B,
    ROUND((b.CAPITALIZATION - a.CAPITALIZATION)/a.CAPITALIZATION * 100, 2) as PCT_CHANGE
FROM barra_factors a
JOIN barra_2008_pk b ON a.BARRID = b.BARRID
WHERE a.NONESTU = 0
    AND a.CAPITALIZATION > 1e9
ORDER BY PCT_CHANGE ASC
LIMIT 20;
""")
# Execute and display results
results = pd.read_sql(comparison_query, engine)
results

## Other JOINS

- RIGHT JOIN keeps all rows from the right table, matches from left
  * Not supported in SQLite, but available in other SQLs
- LEFT JOIN keeps all rows from the left table, matches from right
- FULL OUTER JOIN - shows companies in either year
  * Not supported in SQLite, but available in other SQLs


![](https://static.wixstatic.com/media/2f98e9_c5999377496a4e399bbc042ffe3280fc~mv2.jpg/v1/fill/w_350,h_296,al_c,q_80,usm_0.66_1.00_0.01,enc_avif,quality_auto/2f98e9_c5999377496a4e399bbc042ffe3280fc~mv2.jpg)






In [ ]:
# Inner Join - introduces survivorship bias as no "DEAD" companies.
comparison_query = text("""
SELECT
    a.NAME as Company_2007,
    COALESCE(b.NAME, '**DEAD**') as Company_2008_Status,
    b.CAPITALIZATION/1e9 as CAP_2008_B,
    a.LEVERAGE as LEVERAGE_2007,
    b.LEVERAGE as LEVERAGE_2008
FROM barra_factors a
JOIN barra_2008_pk b ON a.BARRID = b.BARRID
WHERE a.NONESTU = 0
ORDER BY a.LEVERAGE DESC
LIMIT 30;

""")
# Execute and display results
results = pd.read_sql(comparison_query, engine)
results

In [ ]:
# Left Join
# Right Join not available in SqlLite -- Think: how can we do it?

comparison_query = text("""
SELECT
    a.NAME as Company_2007,
    COALESCE(b.NAME, '**DEAD**') as Company_2008_Status,
    b.CAPITALIZATION/1e9 as CAP_2008_B,
    a.LEVERAGE as LEVERAGE_2007,
    b.LEVERAGE as LEVERAGE_2008
FROM barra_factors a                                  -- This is the LEFT table (comes first)
LEFT JOIN barra_2008_pk b ON a.BARRID = b.BARRID      -- This is the RIGHT table (comes second)
WHERE a.NONESTU = 0
ORDER BY a.LEVERAGE DESC
LIMIT 30;

""")
# Execute and display results
results = pd.read_sql(comparison_query, engine)
results

In [ ]:
# Outer Join not available in SqlLite -- how can we do it?

comparison_query = text("""

-- Using ROW_NUMBER() window function to get 10 of each category
WITH categorized_companies AS (
    SELECT
        COALESCE(a.BARRID, b.BARRID) as BARRID,
        a.NAME as Company_2007,
        b.NAME as Company_2008,
        a.LEVERAGE as LEVERAGE_2007,
        b.LEVERAGE as LEVERAGE_2008,
        CASE
            WHEN a.BARRID IS NULL THEN 'New in 2008'
            WHEN b.BARRID IS NULL THEN 'Disappeared'
            ELSE 'Survived'
        END as Status
    FROM barra_factors a
    LEFT JOIN barra_2008_pk b ON a.BARRID = b.BARRID
    WHERE a.NONESTU = 0

    UNION

    SELECT
        COALESCE(a.BARRID, b.BARRID) as BARRID,
        a.NAME as Company_2007,
        b.NAME as Company_2008,
        a.LEVERAGE as LEVERAGE_2007,
        b.LEVERAGE as LEVERAGE_2008,
        'New in 2008' as Status
    FROM barra_2008_pk b
    LEFT JOIN barra_factors a ON a.BARRID = b.BARRID
    WHERE b.NONESTU = 0 AND a.BARRID IS NULL
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY Status
            ORDER BY COALESCE(LEVERAGE_2007, LEVERAGE_2008) DESC
        ) as rn
    FROM categorized_companies
)
SELECT
    Status,
    Company_2007,
    Company_2008,
    ROUND(LEVERAGE_2007, 3) as LEV_2007,
    ROUND(LEVERAGE_2008, 3) as LEV_2008
FROM ranked
WHERE rn <= 10
ORDER BY Status, rn;
""")

results = pd.read_sql(comparison_query, engine)
results

## Non-survivors (LEFT JOIN)

# Finding Missing Companies: Understanding the SQL Query

## Purpose
This query finds companies that existed in 2007 but disappeared in 2008 (possibly due to bankruptcy, mergers, or delistings), focusing on the most leveraged companies.

## Query Breakdown

### 1. Select Statement
```sql
SELECT
    a.BARRID,                                   -- Company identifier
    a.NAME,                                     -- Company name
    a.INDNAME1,                                 -- Primary industry
    ROUND(a.CAPITALIZATION/1e9, 2) as CAP_2007_B,  -- Market cap in billions
    a.LEVERAGE as LEVERAGE_2007                 -- Leverage ratio
```

### 2. Table Join
```sql
FROM barra_factors a                           -- 2007 data (alias 'a')
LEFT JOIN barra_2008_pk b                      -- 2008 data (alias 'b')
    ON a.BARRID = b.BARRID                     -- Matching on company ID
```
- LEFT JOIN keeps ALL 2007 records
- Matches with 2008 where possible
- NULLs where no match exists

### 3. Filters
```sql
WHERE b.BARRID IS NULL                         -- No match in 2008
    AND a.NONESTU = 0                          -- In estimation universe
```
- `b.BARRID IS NULL` finds disappeared companies
- `NONESTU = 0` focuses on significant stocks

### 4. Sorting and Limit
```sql
ORDER BY LEVERAGE_2007 DESC                    -- Highest leverage first
LIMIT 20;                                      -- Show top 20
```

## Example Results
```
BARRID  NAME        INDNAME1  CAP_2007_B  LEVERAGE_2007
LEH     Lehman     BANKS     45.2        2.89
BSC     Bear       BANKS     12.1        2.45
...
```


In [ ]:
# Find companies that existed in 2007 but not in 2008
missing_companies_query = text("""
SELECT
    a.BARRID,
    a.NAME,
    a.INDNAME1,
    ROUND(a.CAPITALIZATION/1e9, 2) as CAP_2007_B,
    a.LEVERAGE as LEVERAGE_2007
FROM barra_factors a
LEFT JOIN barra_2008_pk b ON a.BARRID = b.BARRID
WHERE b.BARRID IS NULL  -- This finds records where there's no match in 2008
    AND a.NONESTU = 0   -- Only look at estimation universe
ORDER BY LEVERAGE_2007 DESC
LIMIT 20;
""")

# Execute and display results
missing_companies = pd.read_sql(missing_companies_query, engine)
print("\nCompanies Present in 2007 but Missing in 2008:")
missing_companies

## Biggest Market Cap drops - LEFT JOIN to include DEAD companies.

In [ ]:

# Left Join
# Right Join not available in SqlLite -- Think: how can we do it?

comparison_query = text("""
SELECT
    a.BARRID,
    a.NAME as Company_2007,
    COALESCE(b.NAME, '**DEAD**') as Company_2008_Status,
    a.INDNAME1,
    ROUND(a.CAPITALIZATION/1e9, 2) as CAP_2007_B,
    ROUND(b.CAPITALIZATION/1e9, 2) as CAP_2008_B,
    ROUND((b.CAPITALIZATION - a.CAPITALIZATION)/a.CAPITALIZATION * 100, 2) as PCT_CHANGE,
    a.LEVERAGE as LEVERAGE_2007,
    b.LEVERAGE as LEVERAGE_2008
FROM barra_factors a                                  -- This is the LEFT table (comes first)
LEFT JOIN barra_2008_pk b ON a.BARRID = b.BARRID      -- This is the RIGHT table (comes second)
WHERE a.CAPITALIZATION > 10e9
ORDER BY a.LEVERAGE DESC
LIMIT 30;

""")

# Execute and display results
results = pd.read_sql(comparison_query, engine)
results

In [ ]:
view_query = text("""
CREATE VIEW IF NOT EXISTS big_losers_view AS
SELECT
    a.BARRID,
    a.NAME as Company_2007,
    COALESCE(b.NAME, '**DEAD**') as Company_2008_Status,
    a.INDNAME1,
    ROUND(a.CAPITALIZATION/1e9, 2) as CAP_2007_B,
    ROUND(b.CAPITALIZATION/1e9, 2) as CAP_2008_B,
    ROUND((b.CAPITALIZATION - a.CAPITALIZATION)/a.CAPITALIZATION * 100, 2) as PCT_CHANGE,
    a.LEVERAGE as LEVERAGE_2007,
    b.LEVERAGE as LEVERAGE_2008
FROM barra_factors a                                  -- This is the LEFT table (comes first)
LEFT JOIN barra_2008_pk b ON a.BARRID = b.BARRID      -- This is the RIGHT table (comes second)
WHERE a.CAPITALIZATION > 10e9
ORDER BY a.LEVERAGE DESC
LIMIT 30;
""")

with engine.connect() as conn:
    conn.execute(view_query)
    conn.commit()

query = 'SELECT * FROM big_losers_view;'
pd.read_sql(query, engine)

## NULL Handling Notes:

- NULL Handling: Working with missing data
- NULL represents unknown or missing values
- NULL comparisons require IS NULL/IS NOT NULL (not = or !=)
- COALESCE replaces NULL with a default value

In [ ]:
query = '''
-- Different ways to handle NULL values
SELECT
    COUNT(*) as total_rows,
    COUNT(PCT_CHANGE) as non_null_PCT_CHANGE,
    COUNT(*) - COUNT(PCT_CHANGE) as null_PCT_CHANGE
FROM big_losers_view;
'''
pd.read_sql(query, engine)


In [ ]:
query = '''
-- COALESCE to replace NULL with default

SELECT *,
    COALESCE(PCT_CHANGE, -99.0) as COALESCE_PCT_CHANGE
FROM big_losers_view
'''
pd.read_sql(query, engine)


- IS NULL vs = NULL (important distinction!)


In [ ]:
query = '''
-- WRONG

SELECT *
FROM big_losers_view
WHERE PCT_CHANGE = NULL
LIMIT 10;
'''
pd.read_sql(query, engine)


In [ ]:
query = '''
-- WRONG

SELECT *
FROM big_losers_view
WHERE PCT_CHANGE <> NULL
LIMIT 10;
'''
pd.read_sql(query, engine)


In [ ]:
query = '''
-- RIGHT

SELECT *
FROM big_losers_view
WHERE PCT_CHANGE is NULL
LIMIT 10;
'''
pd.read_sql(query, engine)


In [ ]:
query = '''
-- RIGHT

SELECT *
FROM big_losers_view
WHERE PCT_CHANGE is NOT NULL
'''
pd.read_sql(query, engine)


# TO BE OR NOT TO BE? (EXISTS and NOT EXISTS)


In [ ]:
comparison_query ='''
-- EXISTS: Efficient existence checks
-- • EXISTS returns true if subquery returns any rows
-- • Often faster than JOIN for existence checks
-- • NOT EXISTS finds missing relationships

-- Find companies that survived from 2007 to 2008
SELECT
    a.NAME,
    a.INDNAME1
FROM barra_factors a
WHERE EXISTS (
    SELECT 1
    FROM barra_2008_pk b
    WHERE b.BARRID = a.BARRID
)
AND a.NONESTU = 0
AND a.INDNAME1 in ('BANKS','SECASSET');

'''

# Execute and display results
results = pd.read_sql(comparison_query, engine)
results

In [ ]:
comparison_query ='''
-- Companies that didn't survive (alternative to LEFT JOIN)
SELECT
    a.NAME,
    a.INDNAME1,
    a.CAPITALIZATION/1e9 as CAP_2007_B
FROM barra_factors a
WHERE NOT EXISTS (
    SELECT 1
    FROM barra_2008_pk b
    WHERE b.BARRID = a.BARRID
)
AND a.NONESTU = 0
AND a.INDNAME1 in ('BANKS','SECASSET')
ORDER BY a.CAPITALIZATION DESC
;
'''

# Execute and display results
results = pd.read_sql(comparison_query, engine)
results


# CASE Statements


In [ ]:
comparison_query ='''
-- CASE: Conditional logic in SQL
-- • CASE allows if-then-else logic within queries
-- • Can be used in SELECT, WHERE, and ORDER BY clauses
-- • Useful for categorizing or transforming data on the fly

SELECT
    NAME,
    CAPITALIZATION/1e9 as CAP_B,
    LEVERAGE,
    CASE
        WHEN CAPITALIZATION > 100e9 THEN 'Mega Cap'
        WHEN CAPITALIZATION > 10e9 THEN 'Large Cap'
        WHEN CAPITALIZATION > 2e9 THEN 'Mid Cap'
        ELSE 'Small Cap'
    END as Size_Category,
    CASE
        WHEN LEVERAGE > 3 THEN 'High Risk'
        WHEN LEVERAGE > 2 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END as Risk_Category
FROM barra_factors
WHERE INDNAME1 = 'BANKS' AND NONESTU = 0
ORDER BY CAPITALIZATION DESC;

'''

# Execute and display results
results = pd.read_sql(comparison_query, engine)
results


# HAVING vs WHERE


In [ ]:
comparison_query ='''
-- HAVING vs WHERE: Filtering at different stages
-- • WHERE filters individual rows before grouping
-- • HAVING filters aggregated results after grouping
-- • Can't use aggregate functions in WHERE clause

-- Find industries with average leverage > 2
SELECT
    INDNAME1,
    COUNT(*) as company_count,
    AVG(LEVERAGE) as avg_leverage
FROM barra_factors
WHERE NONESTU = 0  -- WHERE: filters individual rows
AND LEVERAGE > 2   -- Averages the LEVERAGED companies in each group
GROUP BY INDNAME1
ORDER BY avg_leverage DESC;
'''

# Execute and display results
results = pd.read_sql(comparison_query, engine)
results


In [ ]:
comparison_query ='''
-- HAVING vs WHERE: Filtering at different stages
-- • WHERE filters individual rows before grouping
-- • HAVING filters aggregated results after grouping
-- • Can't use aggregate functions in WHERE clause

-- Find industries with average leverage > 2
SELECT
    INDNAME1,
    COUNT(*) as company_count,
    AVG(LEVERAGE) as avg_leverage
FROM barra_factors
WHERE NONESTU = 0  -- WHERE: filters individual rows
GROUP BY INDNAME1
HAVING AVG(LEVERAGE) > 2  -- HAVING: filters grouped results
ORDER BY avg_leverage DESC;
'''

# Execute and display results
results = pd.read_sql(comparison_query, engine)
results


#LIKE Operator
LIKE is used for pattern matching in SQL, allowing you to search for partial matches in text. It uses two special characters:
- `%` matches any sequence of characters
- `_` matches any single character

## Breaking Down Our Query
```sql
SELECT
    BARRID,
    NAME,
    INDNAME1,
    ROUND(CAPITALIZATION/1e9, 2) as CAP_B,
    LEVERAGE
FROM barra_factors
WHERE (NAME LIKE '%LEHMAN%' OR NAME LIKE '%LEH%')
    AND NONESTU = 0;
```

## Pattern Matching Examples

### 1. `NAME LIKE '%LEHMAN%'`
- Matches: "LEHMAN BROTHERS", "LEHMAN CORP", "NEW LEHMAN INC"
- The `%` on both sides means "find LEHMAN anywhere in the name"

### 2. `NAME LIKE '%LEH%'`
- Matches: "LEH", "LEHMAN", "LEHBROS"
- Broader search to catch abbreviations

## Common LIKE Patterns
```sql
LIKE 'ABC%'    -- Starts with ABC
LIKE '%XYZ'    -- Ends with XYZ
LIKE '%PQR%'   -- Contains PQR
LIKE 'A_C'     -- Three letters, A and C with any character between
```

## Using OR in the Search
```sql
WHERE (NAME LIKE '%LEHMAN%' OR NAME LIKE '%LEH%')
```
- Matches either pattern
- Parentheses group the conditions
- More flexible search


In [ ]:
# First, let's check if Lehman might be under a different name or BARRID
lehman_search_query = text("""
SELECT
    BARRID,
    NAME,
    INDNAME1,
    ROUND(CAPITALIZATION/1e9, 2) as CAP_B,
    LEVERAGE
FROM barra_factors
WHERE (NAME LIKE '%LEHMAN%' OR NAME LIKE '%LEH%')
    AND NONESTU = 0;
""")


# Execute both queries
print("Searching for Lehman Brothers:")
print(pd.read_sql(lehman_search_query, engine))


In [ ]:
lehman_comparison_query = text("""
SELECT
    a.BARRID,
    a.NAME,
    a.INDNAME1,
    ROUND(a.CAPITALIZATION/1e9, 2) as CAP_2007_B,
    ROUND(b.CAPITALIZATION/1e9, 2) as CAP_2008_B,
    ROUND((b.CAPITALIZATION - a.CAPITALIZATION)/a.CAPITALIZATION * 100, 2) as CAP_CHG_PCT,
    ROUND(a.LEVERAGE, 3) as LEVERAGE_2007,
    ROUND(b.LEVERAGE, 3) as LEVERAGE_2008,
    a.VOLTILTY as VOLATILITY_2007,
    b.VOLTILTY as VOLATILITY_2008
FROM barra_factors a
LEFT JOIN barra_2008_pk b ON a.BARRID = b.BARRID
WHERE a.BARRID = 'USARHS2'  -- Lehman's BARRID
;
""")

# Execute and display results
lehman_data = pd.read_sql(lehman_comparison_query, engine)
print("\nLehman Brothers Metrics Comparison:")
print(lehman_data)

# Advanced Research Questions: The Global Financial Crisis Through Barra Data

## Core Focus: Crisis Risk Dynamics
For all queries, remember to only use `a.NONESTU = 0` (2007 estimation universe) to avoid survivorship bias.

## Research Questions

1. **Financial Sector Collapse**
   ```sql
   -- Example structure comparing financial firms
   SELECT
       a.NAME,
       a.INDNAME1,
       a.VOLTILTY as VOL_2007,
       b.VOLTILTY as VOL_2008,
       a.LEVERAGE as LEV_2007
   FROM barra_factors a
   LEFT JOIN barra_2008_pk b ON a.BARRID = b.BARRID
   WHERE a.INDNAME1 IN ('BANKS', 'THRIFTS', 'SECASSET')
   AND a.NONESTU = 0
   ORDER BY a.LEVERAGE DESC;
   ```
   * Did high leverage in 2007 predict bank failures?
   * Which banks had highest volatility increases?
   * Were there warning signs in risk metrics?

2. **Auto Industry Crisis**
   * How did Ford's risk metrics compare to other automakers?
   * Did MOTORVEH companies show similar volatility patterns?
   * Was the auto sector crisis visible in BETA changes?

3. **Government Intervention Analysis**
   ### Question 3.1: Bailout Recipients
   * Compare Fannie Mae, Freddie Mac, and AIG metrics
   * Track their LEVERAGE and VOLTILTY changes
   * Did their BETAs signal systemic importance?

   ### Question 3.2: Industry Impact
   * How did FINSVCS vs BANKS risk metrics differ?
   * Which sectors showed contagion effects?
   * Impact on real estate related sectors (EQTYREIT)?

4. **Size and Value Effects**
   ### Question 4.1: Large vs Small Banks
   * Did size help banks survive?
   * Compare risk metrics across size groups
   * VALUE scores of failed vs surviving banks

   ### Question 4.2: Cross-Industry Size Impact
   * Were larger firms more resilient across all sectors?
   * Did small caps (SC600) show different patterns?
   * Size factor evolution during crisis

5. **Real Economy Impact**
   ### Question 5.1: Construction and Housing
   * Track CONSTRUC and EQTYREIT sector metrics
   * Compare with pre-crisis levels
   * Links between real estate and financial firms

   ### Question 5.2: Consumer Sectors
   * How did consumer discretionary (CLOTHING, RETAIL) fare?
   * Impact on luxury vs basic goods sectors
   * Signs of consumer stress in sector metrics

## Special Focus Areas
1. **Bear Stearns and Lehman Timeline**
   * Risk metric progression before collapse
   * Industry-wide impact after each failure
   * Contagion effects on surviving firms

2. **Real Estate Connection**
   * Track REITs and construction firms
   * Mortgage-related companies performance
   * Cross-sector risk transmission

3. **Government Action Impact**
   * Risk metrics before/after key interventions
   * Sector-wide changes post-bailouts
   * Differences between aided and non-aided firms

If you need additional data to answer these questions, try getting from yfinance, checking that the name matches (not just the ticker).

## Hints for project:

* For example, you could have one table for 2007 Barra USE3 data, one table for 2008 Barra USE3 data, and another table that maps Barra Industries to Barra Sectors. Then you can demonstrate your SQL expertise by joining the tables together in a query to determine the GFC impact on financial sector from 2007 to 2008.
* The mapping for industry to sectors is found on PDF pages 83-84 of the Barra US Equity Model (USE3) handbook
* For simplicity, you can assume a stock is in INDNAME1 (and ignore the less significant INDNAME2 etc.)

In [ ]:
indnames=barra_df[['IND1','INDNAME1']].sort_values('IND1').drop_duplicates()['INDNAME1'].to_list()
print(indnames)

ind2sector={}
for i in indnames:
  if i in ['LIFEINS', 'PRPTYINS', 'BANKS', 'THRIFTS', 'SECASSET', 'FINSVCS']:
    ind2sector[i]='FINANCIALS'
  else:
    ind2sector[i]='UNKNOWN' #Student project TODO (see PDF)


In [ ]:
# Create sector mapping table
create_sector_table = text("""
CREATE TABLE ind_to_sectors (
    INDNAME TEXT PRIMARY KEY,
    SECTOR TEXT
);
""")

# Execute table creation
with engine.connect() as conn:
    conn.execute(create_sector_table)
    conn.commit()

# Insert data from Python dictionary
insert_sector = text("""
INSERT INTO ind_to_sectors (INDNAME, SECTOR) VALUES (:indname, :sector)
""")

# Convert dictionary to list of dictionaries for insertion
sector_data = [{'indname': ind, 'sector': sector}
               for ind, sector in ind2sector.items()
               ]

# Insert the data
with engine.connect() as conn:
    for row in sector_data:
        conn.execute(insert_sector, row)
    conn.commit()

# Verify the data
verify_query = text("""
SELECT * FROM ind_to_sectors
ORDER BY SECTOR, INDNAME;
""")

# Show results
print("\nSector Mapping Table:")
print(pd.read_sql(verify_query, engine))

In [ ]:
# Modified query to include sectors
sector_comparison_query = text("""
SELECT
    a.BARRID,
    a.NAME,
    a.INDNAME1,
    s.SECTOR,
    ROUND(a.CAPITALIZATION/1e9, 2) as CAP_2007_B,
    ROUND(b.CAPITALIZATION/1e9, 2) as CAP_2008_B,
    ROUND((b.CAPITALIZATION - a.CAPITALIZATION)/a.CAPITALIZATION * 100, 2) as PCT_CHANGE
FROM barra_factors a
JOIN barra_2008_pk b ON a.BARRID = b.BARRID
JOIN ind_to_sectors s ON a.INDNAME1 = s.INDNAME
WHERE a.NONESTU = 0
    AND a.CAPITALIZATION > 1e9
    AND s.SECTOR = 'FINANCIALS'
ORDER BY PCT_CHANGE ASC
LIMIT 20;
""")

# Execute and display results
results = pd.read_sql(sector_comparison_query, engine)
print("\nFinancial Companies with Largest Market Cap Declines:")
print(results)

In [ ]:
# Modified query to include sectors
sector_comparison_query = text("""
SELECT *
FROM barra_factors a
LIMIT 20;
""")

# Execute and display results
results = pd.read_sql(sector_comparison_query, engine)
print("\nFinancial Companies with Largest Market Cap Declines:")
results.columns